In [ ]:
# =====================================================================================
# A small open-weight LLM agent for early warning of ICU-onset sepsis under strict
# time-gating: a leakage-controlled replay study in MIMIC-III
# Analysis code (Saxena A, Sinha RK). Released for code availability / peer review.
#
# DATA: no MIMIC-III data, derived data or outputs are included in this notebook.
#   MIMIC-III v1.4 requires credentialed PhysioNet access and a data use agreement
#   (https://physionet.org/content/mimiciii/1.4/). Files written to OUT_DIR (agent traces,
#   evaluation samples, feature tables) contain patient-level data and must not be shared.
#   All LLM inference runs on a local vLLM server; no data are sent to external services.
#
# REQUIREMENTS: Python 3.12, duckdb, pandas, numpy, scikit-learn, joblib, openai (client
#   only, pointed at the local server), matplotlib, pyarrow, openpyxl (optional);
#   MIMIC-III loaded into a DuckDB file (DB_PATH); a vLLM server with tool calling serving
#   nvidia/Qwen3.6-35B-A3B-NVFP4 (VLLM_URL), needed only for Parts 7-8, 11 and 14.
#
# RUN ORDER: cells top to bottom. Long LLM runs are resumable (results append to JSONL).
#   Part 0   configuration, helpers, database connection
#   Part 1-5 cohort, suspected infection, hourly SOFA, Sepsis-3 labels, decision points
#   Part 6-8 time-gated tools + leakage test, agent loop, main batch run
#   Part 9   evaluation (discrimination, alerts, bootstrap CIs, faithfulness)
#   Part 11  paired ablations (Part 10 held planning notes only and has no code)
#   Part 12  extended evaluation without LLM calls (A: features/models; B: comparisons)
#   Part 13  label sensitivity analyses
#   Part 14  new agent runs (A: repeats, trends, hybrid; B: replication, trajectories)
#   Part 15  temporal, leave-one-unit-out and subgroup validation
#   Part 16  collect result tables
# =====================================================================================

# =====================================================================================
# PART 0. Configuration, helper functions and database connection
# Paths, local vLLM endpoint, study-design constants, LLM sampling settings, MIMIC item
# lists; opens the work database with MIMIC-III attached read-only.
# =====================================================================================
# ---- Part 0, configuration ----
import os, json, time, re, math, random, threading
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import duckdb, numpy as np, pandas as pd

# ---- paths ----
DB_PATH = os.path.expanduser(os.environ.get("MIMIC_DB", "~/mimic_project/mimic3.duckdb"))
OUT_DIR = os.path.expanduser("~/mimic_project/agent_runs")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- vLLM server ----
VLLM_URL = "http://localhost:8001/v1"
MODEL    = "nvidia/Qwen3.6-35B-A3B-NVFP4"

# ---- study design ----
MIN_ONSET_HR        = 6      # exclude sepsis already present in the first 6 ICU hours
DECISION_EVERY_H    = 4      # decision point spacing
HORIZON_H           = 12     # label = Sepsis-3 onset within next 12 h
HIDE_INFECTION_DATA = True   # main experiment: agent cannot see antibiotics/cultures (they define the label)
ENABLE_SOFA_TOOL    = False  # a SOFA tool leaks the label logic (SOFA>=2 is half the definition). Use as an ablation.
NOTE_LAG_HOURS      = 6      # notes become visible this many hours after charttime
THINKING            = True   # Qwen thinking mode (compare on/off as an ablation)
INTUBATED_GCS       = "as_charted"  # "as_charted" (verbal ET/Trach scored as charted) or "exclude" (drop those GCS rows; sensitivity analysis)
DEFAULT_WEIGHT_KG   = 80     # used to convert CareVue mcg/min vasopressor rates when no weight is charted

# ---- LLM sampling ----
MAX_TOKENS = 8192            # thinking traces can be long; 3500 truncated some answers before the JSON
# Qwen recommends sampling (not greedy) in thinking mode; greedy can cause repetition loops.
SAMPLING = (dict(temperature=0.6, top_p=0.95, extra={"top_k": 20}) if THINKING
            else dict(temperature=0.7, top_p=0.8, extra={"top_k": 20}))

# ---- evaluation sample (LLM calls are the expensive part) ----
N_EVAL_STAYS_PER_CLASS = 100
POINTS_PER_STAY        = 3
CONCURRENCY            = 8
REBUILD_EVAL_SAMPLE    = False  # False: reuse eval_points.csv if it exists, so agent results stay matched to one sample
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- item lists shared by Parts 2-9 (defined here so ablation re-runs can skip Parts 1-5) ----
ABX = ('cillin|cef|penem|vancomycin|azithromycin|ciprofloxacin|levofloxacin|moxifloxacin|'
       'metronidazole|zosyn|gentamicin|tobramycin|amikacin|clindamycin|linezolid|doxycycline|'
       'trimethoprim|bactrim|sulfamethoxazole|aztreonam|colistin|daptomycin|tigecycline|'
       'erythromycin|unasyn|augmentin')
URINE = (40055, 43175, 40069, 40094, 40715, 40473, 40085, 40057, 40056, 40405, 40428,
         40086, 40096, 40651, 226559, 226560, 226561, 226584, 226563, 226564, 226565, 226567, 226557, 226558)

# helper: SQL expression turning an event time into the first grid hour where it is known
def HR(ts, intime="c.intime"):
    return f"GREATEST(0, CAST(CEIL(date_diff('second', {intime}, {ts}) / 3600.0) AS INTEGER))"

print("DB:", DB_PATH)

# ---- database isolation (v3.1): this notebook NEVER writes to, or locks for writing, the MIMIC-III file ----
# Everything the notebook creates (cohort, SOFA, labels, features) lives in its own file, WORK_DB.
# MIMIC-III is attached READ-ONLY under the name `mimic`, and every MIMIC table is exposed inside WORK_DB as a
# view with the same name, so all SQL in this notebook is unchanged. Other notebooks can read mimic3.duckdb
# at the same time.
# DuckDB rule: while ANY process has mimic3.duckdb open for WRITING, no other process can open it, not even
# read-only. If your other notebook writes to mimic3.duckdb, set USE_SNAPSHOT = True: this notebook then makes a
# private copy once (at a moment when nothing is writing) and never touches the original again.
import shutil
SOURCE_DB     = DB_PATH                                           # original MIMIC-III file, only ever opened read-only
WORK_DB       = os.path.expanduser(os.environ.get("SEPSIS_WORK_DB", "~/mimic_project/sepsis_work.duckdb"))
USE_SNAPSHOT  = False                                             # True = full isolation via a private copy (needs disk space)
SNAPSHOT_PATH = os.path.expanduser("~/mimic_project/mimic3_snapshot.duckdb")
DERIVED_TABLES = ["cohort", "cohort_final", "suspected_infection", "hourly_grid", "ev_labs", "sofa_labs", "ev_gcs", "sofa_cns",
                  "ev_map", "stay_weight", "ev_vaso", "sofa_vaso", "sofa_map", "ev_urine", "sofa_urine", "ev_fio2", "ev_pao2",
                  "ev_pf", "ev_vent", "sofa_resp", "sofa_hourly", "sepsis3", "stay_labels", "decision_points"]

def make_snapshot():
    """One-time private copy of the MIMIC-III file. Needs a moment when no process is writing to it."""
    try:
        duckdb.connect(SOURCE_DB, read_only=True).close()       # fails if another process holds a write lock
    except duckdb.IOException as e:
        raise RuntimeError("Cannot copy now: another process is writing to the MIMIC-III file. Retry when it is idle.") from e
    if os.path.exists(SOURCE_DB + ".wal"):
        raise RuntimeError(f"{SOURCE_DB}.wal exists (un-checkpointed changes). Run CHECKPOINT in the notebook that writes to it, then retry.")
    need = os.path.getsize(SOURCE_DB); free = shutil.disk_usage(os.path.dirname(SNAPSHOT_PATH) or ".").free
    if free < need * 1.1:
        raise RuntimeError(f"Not enough disk space for the snapshot: need {need/1e9:.0f} GB, free {free/1e9:.0f} GB")
    tmp = SNAPSHOT_PATH + ".part"
    print(f"copying {need/1e9:.1f} GB -> {SNAPSHOT_PATH} (one time) ...", flush=True)
    shutil.copy2(SOURCE_DB, tmp)
    t = duckdb.connect(tmp, read_only=True); n = t.execute("SELECT COUNT(*) FROM icustays").fetchone()[0]; t.close()
    os.replace(tmp, SNAPSHOT_PATH)
    print(f"snapshot ready ({n:,} ICU stays). The original file is no longer used by this notebook.")

def connect(rebuild_derived=False, memory_limit="28GB", threads=8):
    """Read-write connection to WORK_DB with MIMIC-III attached read-only.
    rebuild_derived=True removes the views to derived tables of an older run so Parts 1-5 can create real tables."""
    src = SNAPSHOT_PATH if USE_SNAPSHOT else SOURCE_DB
    if USE_SNAPSHOT and not os.path.exists(SNAPSHOT_PATH):
        make_snapshot()
    os.makedirs("/tmp/duckdb_tmp", exist_ok=True)
    c = duckdb.connect(WORK_DB)
    c.execute(f"SET memory_limit='{memory_limit}'"); c.execute(f"SET threads={threads}"); c.execute("SET temp_directory='/tmp/duckdb_tmp'")
    try:
        c.execute(f"ATTACH '{src}' AS mimic (READ_ONLY)")
    except duckdb.IOException as e:
        c.close()
        raise RuntimeError("The MIMIC-III file is locked for WRITING by another process, so DuckDB will not let anyone open it. "
                           "Wait for that process, or set USE_SNAPSHOT = True in Part 0 (copies the file once when it is idle).") from e
    own = {r[0] for r in c.execute("SELECT table_name FROM duckdb_tables() WHERE database_name = current_database()").fetchall()}
    if rebuild_derived:
        views = {r[0] for r in c.execute("SELECT view_name FROM duckdb_views() WHERE database_name = current_database() AND NOT internal").fetchall()}
        for t in DERIVED_TABLES:
            if t in views:
                c.execute(f'DROP VIEW "{t}"')
    for (t,) in c.execute("SELECT table_name FROM duckdb_tables() WHERE database_name = 'mimic'").fetchall():
        if t in own or (rebuild_derived and t in DERIVED_TABLES):
            continue                                              # a real table of this notebook takes precedence
        c.execute(f'CREATE OR REPLACE VIEW "{t}" AS SELECT * FROM mimic."{t}"')
    return c

print("work database (written by this notebook):", WORK_DB)
print("MIMIC-III (read-only):", SNAPSHOT_PATH if USE_SNAPSHOT else SOURCE_DB)

# ---- main-run results file (also set in Part 8; defined here so Parts 9-16 work after a restart) ----
RUN_NAME = (f"agent_{'think' if THINKING else 'nothink'}_{'noinf' if HIDE_INFECTION_DATA else 'inf'}"
            f"_{'sofa' if ENABLE_SOFA_TOOL else 'nosofa'}_gcs-{INTUBATED_GCS}")
RES_PATH = f"{OUT_DIR}/{RUN_NAME}.jsonl"

# connection for Parts 1-5: derived tables are written to WORK_DB; MIMIC-III is only read (see Part 0)
try:
    con.close()
except NameError:
    pass
con = connect(rebuild_derived=True, memory_limit="28GB")         # vLLM shares the Spark's memory, keep this modest

tables = sorted(r[0] for r in con.execute("SELECT table_name FROM duckdb_tables() WHERE database_name = 'mimic'").fetchall())
print(len(tables), "MIMIC tables:", tables)
q1 = lambda sql, *a: con.execute(sql, list(a)).fetchone()[0]
for t in ["patients", "admissions", "icustays", "chartevents", "labevents", "noteevents"]:
    print(f"{t:12s} {q1(f'SELECT COUNT(*) FROM {t}'):,}")


In [ ]:
# =====================================================================================
# PART 1. Cohort
# Adults (>=18 y), first ICU stay per patient, ICU length of stay >= 24 h.
# =====================================================================================
# ---- cohort selection ----
con.execute("""
CREATE OR REPLACE TABLE cohort AS
WITH s AS (
  SELECT i.subject_id, i.hadm_id, i.icustay_id, i.intime, i.outtime, i.los,
         i.first_careunit, p.gender,
         (epoch(i.intime) - epoch(p.dob)) / 86400.0 / 365.242 AS age_raw,
         a.hospital_expire_flag,
         ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) AS stay_rank
  FROM icustays i
  JOIN patients  p ON p.subject_id = i.subject_id
  JOIN admissions a ON a.hadm_id = i.hadm_id
)
SELECT *, CASE WHEN age_raw > 89 THEN 90 ELSE age_raw END AS age FROM s
""")
print("All ICU stays:                 ", q1("SELECT COUNT(*) FROM cohort"))
print("  adults:                      ", q1("SELECT COUNT(*) FROM cohort WHERE age_raw >= 18"))
print("  + first ICU stay per patient:", q1("SELECT COUNT(*) FROM cohort WHERE age_raw >= 18 AND stay_rank = 1"))
print("  + ICU LOS >= 1 day (FINAL):  ", q1("SELECT COUNT(*) FROM cohort WHERE age_raw >= 18 AND stay_rank = 1 AND los >= 1"))
con.execute("""
CREATE OR REPLACE TABLE cohort_final AS
SELECT * FROM cohort WHERE age_raw >= 18 AND stay_rank = 1 AND los >= 1
""")
# the pipeline assumes one ICU stay per hadm_id in the cohort
assert q1("SELECT COUNT(*) - COUNT(DISTINCT hadm_id) FROM cohort_final") == 0, "duplicate hadm_id in cohort"


In [ ]:
# =====================================================================================
# PART 2. Suspected infection
# Antibiotic followed by a culture within 72 h, or culture followed by an antibiotic
# within 24 h; timed at the earlier event.
# =====================================================================================
# ABX regex is defined in Part 0
con.execute(f"""
CREATE OR REPLACE TABLE suspected_infection AS
WITH abx AS (
  SELECT c.hadm_id, p.startdate AS abx_time
  FROM prescriptions p JOIN cohort_final c ON p.hadm_id = c.hadm_id
  WHERE regexp_matches(lower(p.drug), '{ABX}')
    AND (p.route IS NULL OR upper(p.route) NOT IN ('OU','OS','OD','AU','AS','AD','TP','IR'))
    AND p.startdate IS NOT NULL
),
cx AS (
  SELECT c.hadm_id, COALESCE(m.charttime, m.chartdate) AS culture_time
  FROM microbiologyevents m JOIN cohort_final c ON m.hadm_id = c.hadm_id
  WHERE COALESCE(m.charttime, m.chartdate) IS NOT NULL
),
pairs AS (
  SELECT a.hadm_id, a.abx_time, x.culture_time, LEAST(a.abx_time, x.culture_time) AS si_time
  FROM abx a JOIN cx x ON a.hadm_id = x.hadm_id
  WHERE (a.abx_time <= x.culture_time AND x.culture_time <= a.abx_time + INTERVAL 72 HOUR)
     OR (x.culture_time <= a.abx_time AND a.abx_time <= x.culture_time + INTERVAL 24 HOUR)
),
ranked AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY hadm_id ORDER BY si_time) AS rn FROM pairs)
SELECT hadm_id, abx_time, culture_time, si_time AS suspected_infection_time FROM ranked WHERE rn = 1
""")
n, si = q1("SELECT COUNT(*) FROM cohort_final"), q1("SELECT COUNT(*) FROM suspected_infection")
print(f"Suspected infection: {si:,} of {n:,} stays ({100*si/n:.1f}%)   [your earlier run: 19,944 / 32,551 = 61.3%]")


In [ ]:
# =====================================================================================
# PART 3. Hourly grid and hourly SOFA
# Six SOFA components (labs, GCS, MAP/vasopressors, urine, PaO2/FiO2 + ventilation)
# as 24-h rolling worst values, CareVue and MetaVision, combined into sofa_hourly.
# =====================================================================================
# ---- Part 3. Hourly grid and hourly SOFA ----
con.execute("""
CREATE OR REPLACE TABLE hourly_grid AS
SELECT icustay_id, hadm_id, subject_id, intime, outtime, hr,
       intime + hr * INTERVAL 1 HOUR AS hour_time
FROM (
  SELECT icustay_id, hadm_id, subject_id, intime, outtime,
         unnest(generate_series(0, CAST(date_diff('hour', intime, outtime) AS INTEGER))) AS hr
  FROM cohort_final
)
""")
print("grid rows:", f"{q1('SELECT COUNT(*) FROM hourly_grid'):,}", "| stays:", f"{q1('SELECT COUNT(DISTINCT icustay_id) FROM hourly_grid'):,}")

# HR() helper is defined in Part 0

# ---- 3a. Labs: platelets, bilirubin, creatinine ----
con.execute(f"""
CREATE OR REPLACE TABLE ev_labs AS
SELECT c.icustay_id, {HR('l.charttime')} AS hr, l.itemid, l.valuenum
FROM labevents l JOIN cohort_final c ON l.hadm_id = c.hadm_id
WHERE l.itemid IN (51265, 50885, 50912) AND l.valuenum IS NOT NULL
  AND l.charttime >= c.intime - INTERVAL 24 HOUR AND l.charttime <= c.outtime
""")
con.execute("""
CREATE OR REPLACE TABLE sofa_labs AS
WITH h AS (
  SELECT icustay_id, hr,
    MIN(CASE WHEN itemid = 51265 THEN valuenum END) AS plt,
    MAX(CASE WHEN itemid = 50885 THEN valuenum END) AS bili,
    MAX(CASE WHEN itemid = 50912 THEN valuenum END) AS creat
  FROM ev_labs GROUP BY 1, 2
),
j AS (
  SELECT g.icustay_id, g.hr, h.plt, h.bili, h.creat
  FROM hourly_grid g LEFT JOIN h ON h.icustay_id = g.icustay_id AND h.hr = g.hr
)
SELECT icustay_id, hr,
  MIN(plt)   OVER w AS platelets_min,
  MAX(bili)  OVER w AS bilirubin_max,
  MAX(creat) OVER w AS creatinine_max
FROM j
WINDOW w AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
""")
for col in ["platelets_min", "bilirubin_max", "creatinine_max"]:
    print(col, f"{q1(f'SELECT 100.0*COUNT({col})/COUNT(*) FROM sofa_labs'):.1f}% of hours have a value")

# ---- 3b. CNS: Glasgow Coma Scale ----
con.execute(f"""
CREATE OR REPLACE TABLE ev_gcs AS
WITH g AS (
  SELECT ce.icustay_id, ce.charttime,
    MAX(CASE WHEN ce.itemid IN (184, 220739) THEN ce.valuenum END) AS eye,
    MAX(CASE WHEN ce.itemid IN (723, 223900) THEN ce.valuenum END) AS verbal,
    MAX(CASE WHEN ce.itemid IN (454, 223901) THEN ce.valuenum END) AS motor,
    MAX(CASE WHEN ce.itemid IN (723, 223900) AND (ce.value ILIKE '%ET/Trach%' OR ce.value ILIKE '%ETT%') THEN 1 ELSE 0 END) AS ett
  FROM chartevents ce
  WHERE ce.itemid IN (184, 220739, 723, 223900, 454, 223901)
    AND (ce.error IS NULL OR ce.error = 0)
    AND ce.icustay_id IN (SELECT icustay_id FROM cohort_final)
  GROUP BY 1, 2
)
SELECT g.icustay_id, {HR('g.charttime')} AS hr, MIN(g.eye + g.verbal + g.motor) AS gcs
FROM g JOIN cohort_final c ON c.icustay_id = g.icustay_id
WHERE g.eye IS NOT NULL AND g.verbal IS NOT NULL AND g.motor IS NOT NULL
  {'AND g.ett = 0' if INTUBATED_GCS == 'exclude' else ''}
  AND g.charttime >= c.intime AND g.charttime <= c.outtime
GROUP BY 1, 2
""")
con.execute("""
CREATE OR REPLACE TABLE sofa_cns AS
WITH j AS (SELECT g.icustay_id, g.hr, e.gcs FROM hourly_grid g
           LEFT JOIN ev_gcs e ON e.icustay_id = g.icustay_id AND e.hr = g.hr)
SELECT icustay_id, hr, MIN(gcs) OVER w AS gcs_min
FROM j WINDOW w AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
""")
print(f"INTUBATED_GCS = {INTUBATED_GCS!r}")
print(f"GCS available in {q1('SELECT 100.0*COUNT(gcs_min)/COUNT(*) FROM sofa_cns'):.1f}% of hours")

# ---- 3c. Cardiovascular: MAP and vasopressors ----
con.execute(f"""
CREATE OR REPLACE TABLE ev_map AS
SELECT ce.icustay_id, {HR('ce.charttime')} AS hr, MIN(ce.valuenum) AS map
FROM chartevents ce JOIN cohort_final c ON c.icustay_id = ce.icustay_id
WHERE ce.itemid IN (456, 52, 6702, 443, 220052, 220181, 225312)
  AND ce.valuenum > 0 AND ce.valuenum < 300 AND (ce.error IS NULL OR ce.error = 0)
  AND ce.charttime >= c.intime AND ce.charttime <= c.outtime
GROUP BY 1, 2
""")
con.execute("""
CREATE OR REPLACE TABLE stay_weight AS
WITH w AS (
  SELECT ce.icustay_id, ce.charttime,
    CASE WHEN ce.itemid IN (3581, 226531) THEN ce.valuenum * 0.45359237   -- lb
         WHEN ce.itemid = 3582            THEN ce.valuenum * 0.0283495231 -- oz
         ELSE ce.valuenum END AS kg
  FROM chartevents ce
  WHERE ce.itemid IN (762, 763, 3580, 3581, 3582, 226512, 224639, 226531)
    AND ce.valuenum > 0 AND (ce.error IS NULL OR ce.error = 0)
    AND ce.icustay_id IN (SELECT icustay_id FROM cohort_final)
)
SELECT c.icustay_id,
  COALESCE(MEDIAN(CASE WHEN w.charttime <= c.intime + INTERVAL 24 HOUR THEN w.kg END), MEDIAN(w.kg)) AS weight_kg
FROM cohort_final c LEFT JOIN w ON w.icustay_id = c.icustay_id AND w.kg BETWEEN 25 AND 300
GROUP BY 1
""")
print(f"weight available for {q1('SELECT 100.0*COUNT(weight_kg)/COUNT(*) FROM stay_weight'):.1f}% of stays "
      f"(median {q1('SELECT MEDIAN(weight_kg) FROM stay_weight'):.1f} kg)")
con.execute(f"""
CREATE OR REPLACE TABLE ev_vaso AS
SELECT m.icustay_id,
  CASE m.itemid WHEN 221662 THEN 'dopamine' WHEN 221653 THEN 'dobutamine' WHEN 221289 THEN 'epinephrine'
                WHEN 221906 THEN 'norepinephrine' WHEN 221749 THEN 'phenylephrine' WHEN 222315 THEN 'vasopressin' END AS drug,
  {HR('m.starttime')} AS start_hr,
  CAST(CEIL(date_diff('second', c.intime, m.endtime) / 3600.0) AS INTEGER) - 1 AS end_hr,
  CAST(m.rate AS DOUBLE) AS rate
FROM inputevents_mv m JOIN cohort_final c ON c.icustay_id = m.icustay_id
WHERE m.itemid IN (221662, 221653, 221289, 221906, 221749, 222315)
  AND m.rate > 0 AND COALESCE(m.statusdescription, '') <> 'Rewritten'
UNION ALL
SELECT v.icustay_id,
  CASE WHEN v.itemid IN (30043, 30307) THEN 'dopamine' WHEN v.itemid IN (30042, 30306) THEN 'dobutamine'
       WHEN v.itemid IN (30044, 30119, 30309) THEN 'epinephrine' WHEN v.itemid IN (30047, 30120) THEN 'norepinephrine'
       WHEN v.itemid IN (30127, 30128) THEN 'phenylephrine' WHEN v.itemid = 30051 THEN 'vasopressin' END AS drug,
  {HR('v.charttime')} AS start_hr,
  {HR('v.charttime')} + 3 AS end_hr,
  CAST(v.rate AS DOUBLE) / CASE WHEN v.itemid IN (30047, 30044) THEN COALESCE(w.weight_kg, {DEFAULT_WEIGHT_KG}) ELSE 1 END AS rate
FROM inputevents_cv v JOIN cohort_final c ON c.icustay_id = v.icustay_id
LEFT JOIN stay_weight w ON w.icustay_id = v.icustay_id
WHERE v.itemid IN (30043, 30307, 30042, 30306, 30044, 30119, 30309, 30047, 30120, 30127, 30128, 30051)
  AND v.rate > 0
""")
con.execute("""
CREATE OR REPLACE TABLE sofa_vaso AS
SELECT g.icustay_id, g.hr,
  MAX(CASE WHEN v.drug = 'dopamine'       THEN v.rate END) AS dopa,
  MAX(CASE WHEN v.drug = 'dobutamine'     THEN v.rate END) AS dobu,
  MAX(CASE WHEN v.drug = 'epinephrine'    THEN v.rate END) AS epi,
  MAX(CASE WHEN v.drug = 'norepinephrine' THEN v.rate END) AS norepi,
  MAX(CASE WHEN v.drug = 'phenylephrine'  THEN v.rate END) AS phenyl,
  MAX(CASE WHEN v.drug = 'vasopressin'    THEN v.rate END) AS vasop
FROM hourly_grid g LEFT JOIN ev_vaso v
  ON v.icustay_id = g.icustay_id AND g.hr BETWEEN v.start_hr AND v.end_hr
GROUP BY 1, 2
""")
con.execute("""
CREATE OR REPLACE TABLE sofa_map AS
WITH j AS (SELECT g.icustay_id, g.hr, e.map FROM hourly_grid g
           LEFT JOIN ev_map e ON e.icustay_id = g.icustay_id AND e.hr = g.hr)
SELECT icustay_id, hr, MIN(map) OVER w AS map_min
FROM j WINDOW w AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
""")
print(f"MAP available in {q1('SELECT 100.0*COUNT(map_min)/COUNT(*) FROM sofa_map'):.1f}% of hours")
print(f"Any vasopressor in {q1('SELECT 100.0*SUM(CASE WHEN COALESCE(dopa,0)+COALESCE(epi,0)+COALESCE(norepi,0)+COALESCE(phenyl,0)+COALESCE(vasop,0)>0 THEN 1 ELSE 0 END)/COUNT(*) FROM sofa_vaso'):.1f}% of hours")

# URINE item list is defined in Part 0
con.execute(f"""
CREATE OR REPLACE TABLE ev_urine AS
SELECT o.icustay_id, {HR('o.charttime')} AS hr, SUM(o.value) AS ml
FROM outputevents o JOIN cohort_final c ON c.icustay_id = o.icustay_id
WHERE o.itemid IN {URINE} AND o.value >= 0 AND o.value < 5000
  AND o.charttime >= c.intime AND o.charttime <= c.outtime
GROUP BY 1, 2
""")
con.execute("""
CREATE OR REPLACE TABLE sofa_urine AS
WITH j AS (SELECT g.icustay_id, g.hr, e.ml FROM hourly_grid g
           LEFT JOIN ev_urine e ON e.icustay_id = g.icustay_id AND e.hr = g.hr)
SELECT icustay_id, hr,
  CASE WHEN hr >= 24 THEN SUM(ml) OVER w END AS urine_24h
FROM j WINDOW w AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
""")
print(f"24 h urine available in {q1('SELECT 100.0*COUNT(urine_24h)/COUNT(*) FROM sofa_urine'):.1f}% of hours")

# ---- 3e. Respiratory: PaO2/FiO2 and ventilation ----
con.execute(f"""
CREATE OR REPLACE TABLE ev_fio2 AS
SELECT icustay_id, charttime, fio2 FROM (
  SELECT c.icustay_id, ce.charttime,
         CASE WHEN ce.valuenum > 1 THEN ce.valuenum / 100.0 ELSE ce.valuenum END AS fio2
  FROM chartevents ce JOIN cohort_final c ON c.icustay_id = ce.icustay_id
  WHERE ce.itemid IN (223835, 3420, 190) AND ce.valuenum > 0 AND (ce.error IS NULL OR ce.error = 0)
  UNION ALL
  SELECT c.icustay_id, l.charttime,
         CASE WHEN l.valuenum > 1 THEN l.valuenum / 100.0 ELSE l.valuenum END
  FROM labevents l JOIN cohort_final c ON c.hadm_id = l.hadm_id
  WHERE l.itemid = 50816 AND l.valuenum > 0
) WHERE fio2 BETWEEN 0.21 AND 1.0
""")
con.execute("""
CREATE OR REPLACE TABLE ev_pao2 AS
SELECT c.icustay_id, l.charttime, l.valuenum AS pao2
FROM labevents l JOIN cohort_final c ON c.hadm_id = l.hadm_id
WHERE l.itemid = 50821 AND l.valuenum > 0 AND l.valuenum < 700
  AND l.charttime >= c.intime - INTERVAL 24 HOUR AND l.charttime <= c.outtime
""")
con.execute(f"""
CREATE OR REPLACE TABLE ev_pf AS
SELECT p.icustay_id, {HR('p.charttime', 'c.intime')} AS hr, MIN(p.pao2 / f.fio2) AS pf
FROM ev_pao2 p
ASOF JOIN ev_fio2 f ON p.icustay_id = f.icustay_id AND p.charttime >= f.charttime
JOIN cohort_final c ON c.icustay_id = p.icustay_id
WHERE p.charttime - f.charttime <= INTERVAL 4 HOUR
GROUP BY 1, 2
""")
con.execute(f"""
CREATE OR REPLACE TABLE ev_vent AS
SELECT p.icustay_id, {HR('p.starttime')} AS start_hr,
       CAST(CEIL(date_diff('second', c.intime, p.endtime) / 3600.0) AS INTEGER) - 1 AS end_hr
FROM procedureevents_mv p JOIN cohort_final c ON c.icustay_id = p.icustay_id
WHERE p.itemid = 225792
UNION ALL
SELECT ce.icustay_id, {HR('ce.charttime')} AS start_hr, {HR('ce.charttime')} + 5 AS end_hr
FROM chartevents ce JOIN cohort_final c ON c.icustay_id = ce.icustay_id
WHERE ce.itemid IN (720, 223849) AND (ce.error IS NULL OR ce.error = 0)
""")
con.execute("""
CREATE OR REPLACE TABLE sofa_resp AS
WITH j AS (SELECT g.icustay_id, g.hr, e.pf FROM hourly_grid g
           LEFT JOIN ev_pf e ON e.icustay_id = g.icustay_id AND e.hr = g.hr),
w AS (SELECT icustay_id, hr, MIN(pf) OVER w AS pf_min FROM j
      WINDOW w AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)),
v AS (SELECT g.icustay_id, g.hr,
        MAX(CASE WHEN x.icustay_id IS NOT NULL THEN 1 ELSE 0 END) AS vent
      FROM hourly_grid g LEFT JOIN ev_vent x
        ON x.icustay_id = g.icustay_id AND g.hr BETWEEN x.start_hr AND x.end_hr
      GROUP BY 1, 2)
SELECT w.icustay_id, w.hr, w.pf_min, v.vent
FROM w JOIN v ON v.icustay_id = w.icustay_id AND v.hr = w.hr
""")
print(f"P/F available in {q1('SELECT 100.0*COUNT(pf_min)/COUNT(*) FROM sofa_resp'):.1f}% of hours | ventilated {q1('SELECT 100.0*AVG(vent) FROM sofa_resp'):.1f}% of hours")

# ---- 3f. Combine into sofa_hourly ----
con.execute("""
CREATE OR REPLACE TABLE sofa_hourly AS
WITH raw AS (
  SELECT g.icustay_id, g.hadm_id, g.hr, g.hour_time,
    l.platelets_min, l.bilirubin_max, l.creatinine_max,
    n.gcs_min, m.map_min, u.urine_24h, r.pf_min, COALESCE(r.vent, 0) AS vent,
    COALESCE(v.dopa, 0) AS dopa, COALESCE(v.dobu, 0) AS dobu, COALESCE(v.epi, 0) AS epi,
    COALESCE(v.norepi, 0) AS norepi, COALESCE(v.phenyl, 0) AS phenyl, COALESCE(v.vasop, 0) AS vasop
  FROM hourly_grid g
  LEFT JOIN sofa_labs  l ON l.icustay_id = g.icustay_id AND l.hr = g.hr
  LEFT JOIN sofa_cns   n ON n.icustay_id = g.icustay_id AND n.hr = g.hr
  LEFT JOIN sofa_map   m ON m.icustay_id = g.icustay_id AND m.hr = g.hr
  LEFT JOIN sofa_urine u ON u.icustay_id = g.icustay_id AND u.hr = g.hr
  LEFT JOIN sofa_resp  r ON r.icustay_id = g.icustay_id AND r.hr = g.hr
  LEFT JOIN sofa_vaso  v ON v.icustay_id = g.icustay_id AND v.hr = g.hr
),
sc AS (
  SELECT *,
   CASE WHEN pf_min IS NULL THEN 0 WHEN pf_min < 100 AND vent = 1 THEN 4 WHEN pf_min < 200 AND vent = 1 THEN 3
        WHEN pf_min < 300 THEN 2 WHEN pf_min < 400 THEN 1 ELSE 0 END AS s_resp,
   CASE WHEN platelets_min IS NULL THEN 0 WHEN platelets_min < 20 THEN 4 WHEN platelets_min < 50 THEN 3
        WHEN platelets_min < 100 THEN 2 WHEN platelets_min < 150 THEN 1 ELSE 0 END AS s_coag,
   CASE WHEN bilirubin_max IS NULL THEN 0 WHEN bilirubin_max >= 12 THEN 4 WHEN bilirubin_max >= 6 THEN 3
        WHEN bilirubin_max >= 2 THEN 2 WHEN bilirubin_max >= 1.2 THEN 1 ELSE 0 END AS s_liver,
   GREATEST(
     CASE WHEN dopa > 15 OR epi > 0.1 OR norepi > 0.1 THEN 4
          WHEN dopa > 5 OR epi > 0 OR norepi > 0 OR phenyl > 0 OR vasop > 0 THEN 3
          WHEN dopa > 0 OR dobu > 0 THEN 2 ELSE 0 END,
     CASE WHEN map_min IS NOT NULL AND map_min < 70 THEN 1 ELSE 0 END) AS s_cardio,
   CASE WHEN gcs_min IS NULL THEN 0 WHEN gcs_min < 6 THEN 4 WHEN gcs_min < 10 THEN 3
        WHEN gcs_min < 13 THEN 2 WHEN gcs_min < 15 THEN 1 ELSE 0 END AS s_cns,
   GREATEST(
     CASE WHEN creatinine_max IS NULL THEN 0 WHEN creatinine_max >= 5 THEN 4 WHEN creatinine_max >= 3.5 THEN 3
          WHEN creatinine_max >= 2 THEN 2 WHEN creatinine_max >= 1.2 THEN 1 ELSE 0 END,
     CASE WHEN urine_24h IS NULL THEN 0 WHEN urine_24h < 200 THEN 4 WHEN urine_24h < 500 THEN 3 ELSE 0 END) AS s_renal
  FROM raw
)
SELECT *, s_resp + s_coag + s_liver + s_cardio + s_cns + s_renal AS sofa FROM sc
""")
print("rows:", f"{q1('SELECT COUNT(*) FROM sofa_hourly'):,}")
print("\nMean points per component:")
print(con.execute("SELECT ROUND(AVG(s_resp),2) resp, ROUND(AVG(s_coag),2) coag, ROUND(AVG(s_liver),2) liver, ROUND(AVG(s_cardio),2) cardio, ROUND(AVG(s_cns),2) cns, ROUND(AVG(s_renal),2) renal, ROUND(AVG(sofa),2) total FROM sofa_hourly").df().to_string(index=False))

print("\nCardiovascular score distribution (% of hours) [check that 4 is not implausibly common]:")
print(con.execute("SELECT s_cardio, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct FROM sofa_hourly GROUP BY 1 ORDER BY 1").df().to_string(index=False))

# QC: max SOFA in the first 24 h should be higher in patients who died. If not, something is wrong.
print("\nQC: mean of max SOFA (first 24 h) by hospital mortality  [expect survivors lower than non-survivors]")
print(con.execute("""
SELECT c.hospital_expire_flag AS died, COUNT(*) AS n, ROUND(AVG(m.mx), 2) AS mean_max_sofa_24h
FROM (SELECT icustay_id, MAX(sofa) AS mx FROM sofa_hourly WHERE hr <= 24 GROUP BY 1) m
JOIN cohort_final c USING (icustay_id) GROUP BY 1 ORDER BY 1""").df().to_string(index=False))


In [ ]:
# =====================================================================================
# PART 4. Sepsis-3 onset labels
# Suspected infection + SOFA >= 2 within -48/+24 h; onset = later of the two;
# stays classed as no sepsis / onset < 6 h (excluded) / ICU-onset.
# =====================================================================================
# ---- Part 4. Sepsis-3 ground truth ----
con.execute(f"""
CREATE OR REPLACE TABLE sepsis3 AS
WITH q AS (
  SELECT s.icustay_id, s.hour_time, si.suspected_infection_time AS si_time
  FROM sofa_hourly s JOIN suspected_infection si ON si.hadm_id = s.hadm_id
  WHERE s.sofa >= 2
    AND s.hour_time >= si.suspected_infection_time - INTERVAL 48 HOUR
    AND s.hour_time <= si.suspected_infection_time + INTERVAL 24 HOUR
),
f AS (SELECT icustay_id, MIN(hour_time) AS first_sofa_time, ANY_VALUE(si_time) AS si_time FROM q GROUP BY 1)
SELECT f.icustay_id, f.si_time, f.first_sofa_time,
       GREATEST(f.si_time, f.first_sofa_time) AS onset_time,
       CAST(CEIL(date_diff('second', c.intime, GREATEST(f.si_time, f.first_sofa_time)) / 3600.0) AS INTEGER) AS onset_hr
FROM f JOIN cohort_final c USING (icustay_id)
WHERE GREATEST(f.si_time, f.first_sofa_time) <= c.outtime
""")
con.execute(f"""
CREATE OR REPLACE TABLE stay_labels AS
SELECT c.icustay_id, c.hadm_id, c.hospital_expire_flag, s.onset_time, s.onset_hr,
  CASE WHEN s.icustay_id IS NULL THEN 'no_sepsis'
       WHEN s.onset_hr < {MIN_ONSET_HR} THEN 'present_on_admission'
       ELSE 'sepsis_in_icu' END AS stay_class
FROM cohort_final c LEFT JOIN sepsis3 s USING (icustay_id)
""")
print(con.execute("""
SELECT stay_class, COUNT(*) AS n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) AS pct,
       ROUND(100*AVG(hospital_expire_flag), 1) AS mortality_pct, ROUND(MEDIAN(onset_hr), 1) AS median_onset_hr
FROM stay_labels GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print("\nExpect: sepsis stays have clearly higher mortality than no_sepsis stays. Published Sepsis-3 rates in MIMIC are roughly 20-30% of ICU stays.")


In [ ]:
# =====================================================================================
# PART 5. Decision points and stratified evaluation sample
# Points every 4 h from ICU hour 6 until onset; label = onset 1-12 h later;
# 100 + 100 stays, up to 3 points per stay (seed 42).
# =====================================================================================
# ---- Part 5. Decision points and evaluation sample ----
con.execute(f"""
CREATE OR REPLACE TABLE decision_points AS
SELECT s.icustay_id, s.hadm_id, s.hr, s.hour_time, s.sofa, l.onset_hr, l.stay_class,
  CASE WHEN l.onset_hr IS NOT NULL AND l.onset_hr - s.hr BETWEEN 1 AND {HORIZON_H} THEN 1 ELSE 0 END AS y
FROM sofa_hourly s JOIN stay_labels l ON l.icustay_id = s.icustay_id
WHERE s.hr >= {MIN_ONSET_HR} AND s.hr % {DECISION_EVERY_H} = 0
  AND l.stay_class <> 'present_on_admission'
  AND (l.onset_hr IS NULL OR s.hr < l.onset_hr)
""")
print(con.execute("SELECT COUNT(*) AS points, COUNT(DISTINCT icustay_id) AS stays, ROUND(100*AVG(y),2) AS pct_positive FROM decision_points").df().to_string(index=False))

dp = con.execute("SELECT * FROM decision_points").df()
dp.columns = dp.columns.str.lower()          # MIMIC tables have upper-case column names
stays = dp.groupby("icustay_id").agg(stay_class=("stay_class", "first")).reset_index()
rng = np.random.default_rng(SEED)
pos = stays[stays.stay_class == "sepsis_in_icu"].icustay_id.values
neg = stays[stays.stay_class == "no_sepsis"].icustay_id.values
eval_stays = set(rng.choice(pos, min(N_EVAL_STAYS_PER_CLASS, len(pos)), replace=False)) | \
             set(rng.choice(neg, min(N_EVAL_STAYS_PER_CLASS, len(neg)), replace=False))
ev = dp[dp.icustay_id.isin(eval_stays)]
# shuffle, then keep up to POINTS_PER_STAY points per stay
EVAL_CSV = f"{OUT_DIR}/eval_points.csv"
if os.path.exists(EVAL_CSV) and not REBUILD_EVAL_SAMPLE:
    eval_points = pd.read_csv(EVAL_CSV, parse_dates=["hour_time"])
    print(f"Reusing existing {EVAL_CSV} (set REBUILD_EVAL_SAMPLE = True to draw a new one, "
          f"and then delete the old agent .jsonl results)")
else:
    eval_points = (ev.sample(frac=1, random_state=SEED)
                     .groupby("icustay_id").head(POINTS_PER_STAY)
                     .sort_values(["icustay_id", "hr"]).reset_index(drop=True))
    eval_points.to_csv(EVAL_CSV, index=False)
print(f"\nEvaluation sample: {len(eval_points)} decision points from {eval_points.icustay_id.nunique()} stays, "
      f"{eval_points.y.sum()} positive ({100*eval_points.y.mean():.1f}%)")


In [ ]:
# =====================================================================================
# PART 6. Time-gated tool environment and automated leakage test
# (Re)connects read-only, defines Env tools (vitals, labs, urine, notes with 6-h lag)
# filtered to charttime/storetime <= now, and asserts no future timestamps.
# =====================================================================================
# ---- (re)connect to the databases; also the restart point for the analysis parts ----
try:
    con.close()          # connection from Parts 1-5, if open
except NameError:
    pass
con = connect()
q1 = lambda sql, *a: con.execute(sql, list(a)).fetchone()[0]
missing = [t for t in ["cohort_final", "sofa_hourly", "stay_labels", "decision_points"]
           if not con.execute("SELECT COUNT(*) FROM duckdb_views() WHERE view_name = ? UNION ALL SELECT COUNT(*) FROM duckdb_tables() WHERE table_name = ?", [t, t]).df().iloc[:, 0].sum()]
assert not missing, f"tables {missing} not found: run Parts 1-5 once"
eval_points = pd.read_csv(f"{OUT_DIR}/eval_points.csv", parse_dates=["hour_time"])
print("connected (MIMIC-III read-only) |", len(eval_points), "evaluation points loaded")

# ---- Part 6, time-gated tools ----
VITALS = {
  "heart_rate":  [211, 220045],
  "sbp":         [51, 455, 220179, 220050],
  "map":         [456, 52, 6702, 443, 220052, 220181, 225312],
  "resp_rate":   [618, 615, 220210, 224690],
  "temp_c":      [223762, 676],
  "temp_f":      [223761, 678],
  "spo2":        [646, 220277],
}
ITEM2VITAL = {i: k for k, ids in VITALS.items() for i in ids}
LABS = {"lactate": 50813, "wbc": 51301, "creatinine": 50912, "bilirubin": 50885, "platelets": 51265,
        "bicarbonate": 50882, "ph": 50820, "pao2": 50821, "potassium": 50971, "sodium": 50983,
        "bun": 51006, "glucose": 50931, "hemoglobin": 51222, "inr": 51237}
URINE_IDS = list(URINE)
NOTE_CATS = ["Nursing", "Nursing/other", "Physician", "Radiology", "ECG", "Echo", "Respiratory", "General", "Nutrition"]

def _t(x):  # timestamp -> string
    return pd.Timestamp(x).strftime("%Y-%m-%d %H:%M")

def _df(rel):  # DuckDB result -> pandas with lower-case column names (MIMIC tables are upper-case)
    return rel.df().rename(columns=str.lower)

class Env:
    """One patient at one simulated time. All tools read through this object."""
    def __init__(self, cur, icustay_id, now):
        self.cur, self.now = cur, pd.Timestamp(now)
        r = cur.execute("SELECT hadm_id, intime, age, gender, first_careunit FROM cohort_final WHERE icustay_id = ?", [int(icustay_id)]).fetchone()
        self.icustay_id, (self.hadm_id, self.intime, self.age, self.gender, self.unit) = int(icustay_id), r
        self.seen = []        # every timestamp returned to the model (for the leakage test)
        self.seen_notes = []  # note timestamps returned (must respect NOTE_LAG_HOURS)
        self.tool_log = []    # raw tool outputs (for the faithfulness check)

    def _since(self, h): return self.now - pd.Timedelta(hours=float(h))
    def _mark(self, series):
        s = pd.to_datetime(series).dropna()
        if len(s): self.seen.append(s.max())

    # ---------- tools ----------
    def get_vitals(self, hours_back=12):
        hours_back = min(max(float(hours_back), 1), 72)
        df = _df(self.cur.execute("""
            SELECT itemid, charttime, valuenum FROM chartevents
            WHERE icustay_id = ? AND itemid IN (SELECT UNNEST(?)) AND valuenum IS NOT NULL
              AND (error IS NULL OR error = 0)
              AND charttime <= ? AND charttime > ? AND (storetime IS NULL OR storetime <= ?)
        """, [self.icustay_id, list(ITEM2VITAL), self.now, self._since(hours_back), self.now]))
        if df.empty: return {"note": "no vital signs charted in this window"}
        self._mark(df.charttime)
        df["vital"] = df.itemid.map(ITEM2VITAL)
        df.loc[df.vital == "temp_f", "valuenum"] = (df.valuenum - 32) * 5 / 9
        df["vital"] = df.vital.replace({"temp_f": "temp_c"})
        df = df[(df.valuenum > 0) & (df.valuenum < 400)].sort_values("charttime")
        out = {}
        for v, g in df.groupby("vital"):
            out[v] = {"n": len(g), "latest": round(float(g.valuenum.iloc[-1]), 1), "latest_time": _t(g.charttime.iloc[-1]),
                      "min": round(float(g.valuenum.min()), 1), "max": round(float(g.valuenum.max()), 1),
                      "mean": round(float(g.valuenum.mean()), 1),
                      "last_values": [round(float(x), 1) for x in g.valuenum.tail(6)]}
        return out

    def get_labs(self, lab_names, hours_back=48):
        if isinstance(lab_names, str): lab_names = [lab_names]
        hours_back = min(max(float(hours_back), 1), 168)
        out = {}
        for name in lab_names:
            key = str(name).lower().strip()
            if key not in LABS:
                out[name] = f"unknown lab. choose from: {sorted(LABS)}"; continue
            df = _df(self.cur.execute("""
                SELECT charttime, valuenum FROM labevents
                WHERE hadm_id = ? AND itemid = ? AND valuenum IS NOT NULL AND charttime <= ? AND charttime > ?
                ORDER BY charttime DESC LIMIT 6
            """, [self.hadm_id, LABS[key], self.now, self._since(hours_back)]))
            if df.empty: out[key] = "no result in this window"; continue
            self._mark(df.charttime)
            out[key] = [{"time": _t(r.charttime), "value": round(float(r.valuenum), 2)} for r in df.itertuples()]
        return out

    def get_urine_output(self, hours_back=24):
        hours_back = min(max(float(hours_back), 1), 72)
        df = _df(self.cur.execute("""
            SELECT charttime, value FROM outputevents
            WHERE icustay_id = ? AND itemid IN (SELECT UNNEST(?)) AND value >= 0
              AND charttime <= ? AND charttime > ? AND (storetime IS NULL OR storetime <= ?)
        """, [self.icustay_id, URINE_IDS, self.now, self._since(hours_back), self.now]))
        if df.empty: return {"note": "no urine output recorded in this window"}
        self._mark(df.charttime)
        return {"total_ml": round(float(df.value.sum()), 0), "hours_window": hours_back,
                "ml_per_hour": round(float(df.value.sum()) / hours_back, 1), "n_records": len(df),
                "last_record_time": _t(df.charttime.max())}

    def search_notes(self, query, max_results=4):
        max_results = min(max(int(max_results), 1), 6)
        cutoff = self.now - pd.Timedelta(hours=NOTE_LAG_HOURS)
        df = _df(self.cur.execute("""
            SELECT charttime, trim(category) AS category, text FROM noteevents
            WHERE hadm_id = ? AND trim(category) IN (SELECT UNNEST(?))
              AND (iserror IS NULL OR iserror <> 1) AND charttime IS NOT NULL AND charttime <= ?
              AND (storetime IS NULL OR storetime <= ?)
              AND text ILIKE '%' || ? || '%'
            ORDER BY charttime DESC LIMIT ?
        """, [self.hadm_id, NOTE_CATS, cutoff, self.now, str(query), max_results]))
        if df.empty: return {"note": f"no notes matching '{query}' available at this time"}
        self._mark(df.charttime)
        self.seen_notes.extend(pd.to_datetime(df.charttime).tolist())
        res = []
        for r in df.itertuples():
            i = r.text.lower().find(str(query).lower())
            res.append({"time": _t(r.charttime), "category": r.category,
                        "snippet": re.sub(r"\s+", " ", r.text[max(0, i - 200): i + 300])})
        return res

    def get_sofa(self):
        hr = int((self.now - pd.Timestamp(self.intime)).total_seconds() // 3600)
        r = self.cur.execute("""SELECT hour_time, sofa, s_resp, s_coag, s_liver, s_cardio, s_cns, s_renal
                                FROM sofa_hourly WHERE icustay_id = ? AND hr = ?""", [self.icustay_id, hr]).fetchone()
        if r is None: return {"note": "no SOFA available"}
        self._mark([r[0]])
        return dict(zip(["as_of", "sofa_total", "resp", "coagulation", "liver", "cardiovascular", "cns", "renal"],
                        [_t(r[0])] + list(r[1:])))

    def get_infection_data(self):
        a = _df(self.cur.execute("""SELECT startdate, drug FROM prescriptions WHERE hadm_id = ? AND startdate + INTERVAL 1 DAY <= ?
                                    AND regexp_matches(lower(drug), ?) ORDER BY startdate LIMIT 8""",
                                 [self.hadm_id, self.now, ABX]))
        m = _df(self.cur.execute("""SELECT COALESCE(charttime, chartdate) AS t, spec_type_desc, org_name FROM microbiologyevents
                                    WHERE hadm_id = ? AND COALESCE(charttime, chartdate + INTERVAL 1 DAY) <= ? ORDER BY 1 LIMIT 8""",
                                 [self.hadm_id, self.now]))
        if len(a): self._mark(a.startdate)
        if len(m): self._mark(m.t)
        return {"antibiotics": [{"date": _t(r.startdate), "drug": r.drug} for r in a.itertuples()],
                "cultures": [{"time": _t(r.t), "specimen": r.spec_type_desc, "organism": r.org_name} for r in m.itertuples()]}

# ---------- tool schemas for the model ----------
def _fn(name, desc, props, req=()):
    return {"type": "function", "function": {"name": name, "description": desc,
            "parameters": {"type": "object", "properties": props, "required": list(req)}}}
TOOLS = [
  _fn("get_vitals", "Summary of vital signs (heart_rate, sbp, map, resp_rate, temp_c, spo2) charted in the last N hours.",
      {"hours_back": {"type": "number", "description": "look-back window in hours (default 12, max 72)"}}),
  _fn("get_labs", "Recent lab results. Allowed names: " + ", ".join(sorted(LABS)) + ".",
      {"lab_names": {"type": "array", "items": {"type": "string"}}, "hours_back": {"type": "number", "description": "default 48"}}, ["lab_names"]),
  _fn("get_urine_output", "Total urine output over the last N hours.", {"hours_back": {"type": "number", "description": "default 24"}}),
  _fn("search_notes", "Keyword search in clinical notes (nursing, physician, radiology) written before now.",
      {"query": {"type": "string"}, "max_results": {"type": "integer"}}, ["query"]),
]
if ENABLE_SOFA_TOOL:
    TOOLS.append(_fn("get_sofa", "Current SOFA score and its six components.", {}))
if not HIDE_INFECTION_DATA:
    TOOLS.append(_fn("get_infection_data", "Antibiotics and cultures ordered so far.", {}))
print("tools:", [t["function"]["name"] for t in TOOLS])

# ---- Part 6, leakage test ----
cur = con.cursor()
test_pts = eval_points.sample(min(30, len(eval_points)), random_state=1)
violations, lag_violations, n_notes, calls = 0, 0, 0, 0
for r in test_pts.itertuples():
    env = Env(cur, r.icustay_id, r.hour_time)
    env.get_vitals(24); env.get_labs(list(LABS), 72); env.get_urine_output(24)
    for kw in ["sepsis", "fever", "hypotension", "pressors"]: env.search_notes(kw)
    if ENABLE_SOFA_TOOL: env.get_sofa()
    if not HIDE_INFECTION_DATA: env.get_infection_data()
    calls += 1
    bad = [t for t in env.seen if t > env.now]
    violations += len(bad)
    # notes must also respect the safety lag
    lag_cut = env.now - pd.Timedelta(hours=NOTE_LAG_HOURS)
    n_notes += len(env.seen_notes)
    lag_violations += sum(t > lag_cut for t in env.seen_notes)
print(f"{calls} decision points tested, {violations} future timestamps returned")
print(f"{n_notes} note snippets returned, {lag_violations} newer than now - {NOTE_LAG_HOURS} h")
assert violations == 0, "TIME-GATING VIOLATION: fix before running the agent"
assert lag_violations == 0, "NOTE-LAG VIOLATION: fix before running the agent"
print("PASS: no future data is reachable through the tools")
env = Env(cur, test_pts.iloc[0].icustay_id, test_pts.iloc[0].hour_time)
print("\nExample get_vitals at", env.now, ":\n", json.dumps(env.get_vitals(12), indent=1)[:900])


In [ ]:
# =====================================================================================
# PART 7. Agent loop
# System prompt, ReAct-style tool-calling loop (max 8 steps) via the local vLLM
# OpenAI-compatible API, JSON parsing; optional single-case smoke test.
# =====================================================================================
# ---- Part 7, agent loop definitions ----
from openai import OpenAI
client = OpenAI(base_url=VLLM_URL, api_key="EMPTY", timeout=600)

SYSTEM = """You are an ICU clinical decision-support agent. You monitor one adult ICU patient at a given moment.
Task: estimate the probability that this patient will develop sepsis (suspected infection plus acute organ dysfunction) within the NEXT 12 HOURS.
Rules:
- You can only know what the tools return. Query the tools for the data you need. Never guess or invent values.
- Typical useful data: vital-sign trends, lactate, WBC, creatinine, bilirubin, platelets, urine output, blood gases, nursing/physician notes.
- Use few tool calls (usually 3-6). Stop when you have enough evidence.
- Final answer: ONLY a JSON object, no other text:
{"sepsis_risk": <number 0-1>, "alert": <true|false>, "evidence": ["short statement with the actual numbers and times", ...], "missing_data": ["...", ...]}
- Set alert=true only if the risk is high enough that a clinician should be notified now. Alerts that are wrong cause alert fatigue."""

def parse_final(text):
    if not text: return None
    m = re.search(r"\{.*\}", text, re.S)
    if not m: return None
    try:
        d = json.loads(m.group(0))
        d["sepsis_risk"] = float(d["sepsis_risk"]); d["alert"] = bool(d["alert"])
        return d
    except Exception:
        return None

def run_agent(env, max_steps=8, thinking=THINKING):
    user = (f"Patient: {int(env.age)}-year-old {'man' if env.gender == 'M' else 'woman'}, admitted to {env.unit}. "
            f"ICU admission time: {_t(env.intime)}. Current time: {_t(env.now)} "
            f"({(env.now - pd.Timestamp(env.intime)).total_seconds() / 3600:.0f} hours into the ICU stay). "
            "Assess the risk of sepsis onset in the next 12 hours.")
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    trace, n_tools, ptok, ctok, final, t0 = [], 0, 0, 0, None, time.time()
    retried, force_final, n_trunc = False, False, 0
    for step in range(max_steps + 2):              # +1 forced-final step, +1 possible JSON retry
        if step >= max_steps and not force_final:
            force_final = True
            messages.append({"role": "user", "content": "Give your final JSON answer now."})
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, max_tokens=MAX_TOKENS,
            temperature=SAMPLING["temperature"], top_p=SAMPLING["top_p"], seed=SEED + step,
            tools=None if force_final else TOOLS, tool_choice=None if force_final else "auto",
            extra_body={"chat_template_kwargs": {"enable_thinking": thinking}, **SAMPLING["extra"]})
        choice = resp.choices[0]; msg = choice.message
        n_trunc += choice.finish_reason == "length"
        ptok += resp.usage.prompt_tokens; ctok += resp.usage.completion_tokens
        reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
        if not msg.tool_calls:
            final = parse_final(msg.content)
            trace.append({"step": step, "type": "final", "content": msg.content, "reasoning": reasoning,
                          "finish_reason": choice.finish_reason})
            if final is None and not retried:      # one reminder, then give up
                retried, force_final = True, True
                messages.append({"role": "assistant", "content": msg.content or ""})
                messages.append({"role": "user", "content": "Return ONLY the final JSON object in the required format, nothing else."})
                continue
            break
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [{"id": tc.id, "type": "function",
                                         "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                        for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            n_tools += 1
            try:
                args = json.loads(tc.function.arguments or "{}")
                result = getattr(env, tc.function.name)(**args) if tc.function.name in [t["function"]["name"] for t in TOOLS] \
                         else {"error": f"unknown tool {tc.function.name}"}
            except Exception as e:
                args, result = tc.function.arguments, {"error": f"{type(e).__name__}: {e}"}
            out = json.dumps(result, default=str)
            env.tool_log.append(out)
            trace.append({"step": step, "type": "tool", "name": tc.function.name, "args": args, "result": out,
                          "reasoning": reasoning, "finish_reason": choice.finish_reason})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {"final": final, "trace": trace, "n_tool_calls": n_tools, "prompt_tokens": ptok,
            "json_retry": retried, "n_truncated": int(n_trunc),
            "completion_tokens": ctok, "latency_s": round(time.time() - t0, 1), "tool_log": env.tool_log}

def vllm_ready(timeout=10):
    """Quick health check: is the vLLM server up, serving MODEL, and able to answer a 5-token request within 60 s?"""
    import urllib.request
    try:
        with urllib.request.urlopen(VLLM_URL.rstrip("/") + "/models", timeout=timeout) as r:
            models = [m["id"] for m in json.load(r)["data"]]
    except Exception as e:
        print(f"vLLM is not reachable at {VLLM_URL} ({type(e).__name__}: {e}). Start the server, then re-run this cell.")
        return False
    if MODEL not in models:
        print(f"vLLM is up but serves {models}, not {MODEL!r}. Fix MODEL in Part 0 or restart vLLM with that model.")
        return False
    t0 = time.time()
    try:
        client.with_options(timeout=60, max_retries=0).chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "Reply with OK."}], max_tokens=5,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    except Exception as e:
        print(f"vLLM is up but a 5-token request did not finish within 60 s ({type(e).__name__}). "
              "It is probably busy with another job (requests queue behind it). Run the LLM parts when it is free.")
        return False
    print(f"vLLM ready: {MODEL} answered a 5-token request in {time.time() - t0:.1f} s")
    return True
print('agent loop defined (no request sent yet)')

# ---- optional single-case smoke test (needs the vLLM server) ----
if vllm_ready():
    p = eval_points[eval_points.y == 1].iloc[0]
    env = Env(con.cursor(), p.icustay_id, p.hour_time)
    res = run_agent(env)
    print("TRUE LABEL y =", p.y, "| onset in", p.onset_hr - p.hr, "h")
    print("truncated steps:", res["n_truncated"], "| JSON retry used:", res["json_retry"])
    print("tool calls:", res["n_tool_calls"], "| tokens:", res["prompt_tokens"], "+", res["completion_tokens"], "| latency:", res["latency_s"], "s")
    for t in res["trace"]:
        print(f"[{t['step']}] {t['type']:5s}", t.get("name", ""), json.dumps(t.get("args", ""))[:120] if t["type"] == "tool" else (t["content"] or "")[:600])
    print("\nPARSED:", json.dumps(res["final"], indent=1))


In [ ]:
# =====================================================================================
# PART 8. Batch run of the main agent configuration
# Runs the agent on all evaluation points (resumable, results to JSONL).
# =====================================================================================
# ---- Part 8. Batch run ----
RUN_NAME = (f"agent_{'think' if THINKING else 'nothink'}_{'noinf' if HIDE_INFECTION_DATA else 'inf'}"
            f"_{'sofa' if ENABLE_SOFA_TOOL else 'nosofa'}_gcs-{INTUBATED_GCS}")
RES_PATH = f"{OUT_DIR}/{RUN_NAME}.jsonl"
LIMIT = None          # full run (set e.g. 24 for a quick trial)
lock = threading.Lock()

pts = pd.read_csv(f"{OUT_DIR}/eval_points.csv", parse_dates=["hour_time"])
if LIMIT: pts = pts.sample(LIMIT, random_state=SEED)
done = set()
if os.path.exists(RES_PATH):
    for line in open(RES_PATH):
        d = json.loads(line); done.add((d["icustay_id"], d["hr"]))
todo = [r for r in pts.itertuples() if (r.icustay_id, r.hr) not in done]
print(f"{len(done)} done, {len(todo)} to run -> {RES_PATH}")

def work(r):
    try:
        env = Env(con.cursor(), r.icustay_id, r.hour_time)
        out = run_agent(env)
    except Exception as e:
        out = {"final": None, "trace": [{"type": "error", "content": f"{type(e).__name__}: {e}"}], "n_tool_calls": 0,
               "prompt_tokens": 0, "completion_tokens": 0, "latency_s": 0, "tool_log": [],
               "json_retry": False, "n_truncated": 0}
    rec = {"icustay_id": int(r.icustay_id), "hr": int(r.hr), "y": int(r.y), "run": RUN_NAME, **out}
    with lock:
        with open(RES_PATH, "a") as f: f.write(json.dumps(rec, default=str) + "\n")
    return rec

t0 = time.time(); ok = 0
with ThreadPoolExecutor(CONCURRENCY) as ex:
    futs = [ex.submit(work, r) for r in todo]
    for i, f in enumerate(as_completed(futs), 1):
        rec = f.result(); ok += rec["final"] is not None
        if i % 10 == 0 or i == len(futs):
            print(f"{i}/{len(futs)} finished | valid answers {ok} | {(time.time()-t0)/60:.1f} min", flush=True)


In [ ]:
# =====================================================================================
# PART 9. Evaluation
# Agent vs SOFA vs gradient boosting: AUROC/AUPRC, alert performance, stay-level
# bootstrap CIs and paired tests, numeric faithfulness of cited evidence.
# =====================================================================================
# ---- Part 9, load agent results; SOFA-feature set and baseline models (also used by Part 12) ----
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

# ---- load agent results ----
rows = [json.loads(l) for l in open(RES_PATH)]
# keep only the current evaluation sample (the results file can hold points from an older sample)
keep = set(zip(eval_points.icustay_id.astype(int), eval_points.hr.astype(int)))
n_all = len(rows)
rows = [r for r in rows if (int(r["icustay_id"]), int(r["hr"])) in keep]
print(f"results file: {n_all} records | in current evaluation sample: {len(rows)} | dropped: {n_all - len(rows)}"
      f" | sample points without a result: {len(keep) - len(rows)}")
res = pd.DataFrame([{"icustay_id": r["icustay_id"], "hr": r["hr"], "y": r["y"],
                     "valid": r["final"] is not None,
                     "risk": r["final"]["sepsis_risk"] if r["final"] else np.nan,
                     "alert": r["final"]["alert"] if r["final"] else np.nan,
                     "n_tools": r["n_tool_calls"], "ptok": r["prompt_tokens"], "ctok": r["completion_tokens"],
                     "latency_s": r["latency_s"], "retry": r.get("json_retry", False),
                     "trunc": r.get("n_truncated", 0)} for r in rows])
print(f"{len(res)} cases | valid JSON answers: {100*res.valid.mean():.1f}% | mean tool calls: {res.n_tools.mean():.1f} "
      f"| mean tokens: {(res.ptok+res.ctok).mean():.0f} | mean latency: {res.latency_s.mean():.1f}s")
print(f"JSON retry used: {100*res.retry.mean():.1f}% | cases with a truncated step: {100*(res.trunc > 0).mean():.1f}%")
assert res.valid.any(), "no valid agent answers: inspect the traces before evaluating"

# ---- baselines ----
FEATS = ["platelets_min", "bilirubin_max", "creatinine_max", "map_min", "gcs_min", "urine_24h", "pf_min",
         "vent", "dopa", "epi", "norepi", "hr"]
allp = con.execute(f"""SELECT d.icustay_id, d.hr, d.y, d.sofa, {', '.join('s.' + f for f in FEATS if f != 'hr')}
                       FROM decision_points d JOIN sofa_hourly s ON s.icustay_id = d.icustay_id AND s.hr = d.hr""").df()
allp.columns = allp.columns.str.lower()          # MIMIC tables have upper-case column names
test = allp.merge(res[["icustay_id", "hr"]], on=["icustay_id", "hr"])
train = allp[~allp.icustay_id.isin(set(res.icustay_id))]
train = train.sort_values(["icustay_id", "hr"]).reset_index(drop=True)   # fixed order -> reproducible
gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED).fit(train[FEATS], train.y)
test["gb"] = gb.predict_proba(test[FEATS])[:, 1]
df = test.merge(res, on=["icustay_id", "hr", "y"])
dv = df[df.valid]

def auc(y, s):
    try: return roc_auc_score(y, s), average_precision_score(y, s)
    except ValueError: return (np.nan, np.nan)
table = pd.DataFrame({
  "SOFA score":        auc(dv.y, dv.sofa),
  "Gradient boosting": auc(dv.y, dv.gb),
  "Agent":             auc(dv.y, dv.risk)}, index=["AUROC", "AUPRC"]).T.round(3)
print(f"\nEvaluated on {len(dv)} decision points ({dv.y.sum()} positive, {100*dv.y.mean():.1f}%)")
print(table.to_string())

# ---- alert quality: agent's own alert vs SOFA>=2 vs SOFA threshold matched to the agent's alert rate ----
def sens_spec(y, a):
    y, a = np.asarray(y).astype(int), np.asarray(a).astype(int)
    tp, fn, fp, tn = ((a == 1) & (y == 1)).sum(), ((a == 0) & (y == 1)).sum(), ((a == 1) & (y == 0)).sum(), ((a == 0) & (y == 0)).sum()
    return {"alert_rate": round((a == 1).mean(), 3), "sensitivity": round(tp / max(tp + fn, 1), 3),
            "specificity": round(tn / max(tn + fp, 1), 3), "PPV (inflated, enriched sample)": round(tp / max(tp + fp, 1), 3)}
agent_alert = dv.alert.astype(int)
thr = np.quantile(dv.sofa, 1 - max(agent_alert.mean(), 1e-6))
alerts = pd.DataFrame({
  "Agent alert flag": sens_spec(dv.y, agent_alert),
  "SOFA >= 2":        sens_spec(dv.y, dv.sofa >= 2),
  f"SOFA >= {thr:.0f} (matched alert rate)": sens_spec(dv.y, dv.sofa >= thr)}).T
print(alerts.to_string())

# ---- lead time: hours between first agent alert and Sepsis-3 onset (septic stays, evaluated points only) ----
lab = con.execute("SELECT icustay_id, onset_hr FROM stay_labels WHERE onset_hr IS NOT NULL").df()
lab.columns = lab.columns.str.lower()          # MIMIC tables have upper-case column names
al = dv[dv.alert.astype(int) == 1].groupby("icustay_id").hr.min().rename("first_alert_hr").reset_index()
lt = lab.merge(al, on="icustay_id").assign(lead_h=lambda d: d.onset_hr - d.first_alert_hr)
lt = lt[lt.icustay_id.isin(dv.icustay_id)]
n_sep = dv[dv.y == 1].icustay_id.nunique()
print(f"\nLead time: {len(lt)} septic stays with an alert before onset (of {n_sep} with a positive point). "
      f"Median {lt.lead_h.median():.1f} h, IQR {lt.lead_h.quantile(.25):.1f}-{lt.lead_h.quantile(.75):.1f} h")
early = (lt.lead_h > HORIZON_H).sum()
print(f"  of these, {len(lt) - early} first alerts fell inside the {HORIZON_H} h horizon and {early} earlier "
      f"(an alert > {HORIZON_H} h before onset counts as a false positive at that point)")
print("(Only the sampled decision points were evaluated, so this under-estimates the true earliest alert. For a proper "
      "lead-time result, run ALL decision points of a smaller set of septic stays.)")

# ---- 95% CIs by stay-level bootstrap, and paired comparisons between methods ----
# Resample whole ICU stays (not points), because points from the same stay are correlated.
N_BOOT = 2000
dvr = dv.reset_index(drop=True)
y = dvr.y.values.astype(int)
scores = {"SOFA score": dvr.sofa.values, "Gradient boosting": dvr.gb.values, "Agent": dvr.risk.values.astype(float)}
idx_by_stay = list(dvr.groupby("icustay_id").indices.values())
rng_b = np.random.default_rng(SEED)
bs = {(m, k): [] for m in scores for k in ("AUROC", "AUPRC")}
for _ in range(N_BOOT):
    ix = np.concatenate([idx_by_stay[i] for i in rng_b.integers(0, len(idx_by_stay), len(idx_by_stay))])
    yb = y[ix]
    if yb.min() == yb.max(): continue
    for m, s in scores.items():
        bs[(m, "AUROC")].append(roc_auc_score(yb, s[ix]))
        bs[(m, "AUPRC")].append(average_precision_score(yb, s[ix]))
bs = {k: np.array(v) for k, v in bs.items()}

def fmt(point, arr): return f"{point:.3f} ({np.percentile(arr, 2.5):.3f}-{np.percentile(arr, 97.5):.3f})"
ci = pd.DataFrame({m: {k: fmt(table.loc[m, k], bs[(m, k)]) for k in ("AUROC", "AUPRC")} for m in scores}).T
print(f"Point estimate (95% CI), {len(bs[('Agent', 'AUROC')])} stay-level bootstrap resamples, "
      f"{len(idx_by_stay)} stays, {len(dvr)} points, chance AUPRC = {y.mean():.3f}")
print(ci.to_string())

pairs = [("Agent", "SOFA score"), ("Agent", "Gradient boosting"), ("Gradient boosting", "SOFA score")]
comp = []
for a, b in pairs:
    for k in ("AUROC", "AUPRC"):
        d = bs[(a, k)] - bs[(b, k)]
        p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
        comp.append({"comparison": f"{a} - {b}", "metric": k,
                     "difference": round(table.loc[a, k] - table.loc[b, k], 3),
                     "95% CI": f"{np.percentile(d, 2.5):.3f} to {np.percentile(d, 97.5):.3f}",
                     "p (bootstrap)": "<0.001" if p < 1e-3 else round(p, 3)})
comp = pd.DataFrame(comp)
print("\nPaired differences (same bootstrap resamples):")
print(comp.to_string(index=False))

ci.to_csv(f"{OUT_DIR}/{RUN_NAME}_discrimination_ci.csv"); comp.to_csv(f"{OUT_DIR}/{RUN_NAME}_paired_comparisons.csv", index=False)

# ---- faithfulness: do the numbers quoted as evidence appear in the tool outputs? ----
num_re = re.compile(r"(?<![\w.])-?\d+(?:\.\d+)?")
# remove dates/times first, otherwise "2150-03-04 12:00" yields 2150, -3, -4, 12, 0 and inflates matches
ts_re = re.compile(r"\d{4}-\d{2}-\d{2}(?:[ T]\d{1,2}:\d{2}(?::\d{2})?)?|\b\d{1,2}:\d{2}(?::\d{2})?\b")
def numbers(s): return [float(x) for x in num_re.findall(ts_re.sub(" ", s))]
checked = supported = 0
for r in rows:
    if not r["final"]: continue
    tool_nums = np.array(sorted(set(n for t in r["tool_log"] for n in numbers(t))) or [np.nan])
    for ev_text in r["final"].get("evidence", []):
        for n in numbers(str(ev_text)):
            if abs(n) < 1 or n in (12.0, 24.0, 48.0): continue      # skip trivial counts and window sizes
            checked += 1
            supported += bool(np.any(np.isclose(tool_nums, n, rtol=0.01, atol=0.051)))
print(f"Faithfulness: {supported}/{checked} cited numbers ({100*supported/max(checked,1):.1f}%) appear in tool results")
print("Numbers that do NOT appear are hallucination candidates. Read those cases manually.")

# ---- save a summary for the paper ----
table.to_csv(f"{OUT_DIR}/{RUN_NAME}_discrimination.csv"); alerts.to_csv(f"{OUT_DIR}/{RUN_NAME}_alerts.csv")
print("saved to", OUT_DIR)


In [ ]:
# =====================================================================================
# PART 11. Paired ablations
# No thinking, no notes, single prompt, SOFA tool, infection data; same 568 points.
# =====================================================================================
# ---- Part 11, ablation definitions ----
# ---- ablation definitions ----
BASE_TOOLS = ["get_vitals", "get_labs", "get_urine_output", "search_notes"]
ABLATIONS = {
    "nothink":      dict(thinking=False, tools=BASE_TOOLS,                          mode="agent"),
    "nonotes":      dict(thinking=True,  tools=BASE_TOOLS[:3],                      mode="agent"),
    "singleprompt": dict(thinking=True,  tools=[],                                  mode="single"),
    "sofatool":     dict(thinking=True,  tools=BASE_TOOLS + ["get_sofa"],           mode="agent"),
    "infdata":      dict(thinking=True,  tools=BASE_TOOLS + ["get_infection_data"], mode="agent"),
}
RUN_ABLATIONS = ["nothink", "nonotes", "singleprompt", "sofatool", "infdata"]   # remove any you do not want
ABL_LIMIT = None          # e.g. 24 for a quick trial of each ablation

ALL_TOOLS = {t["function"]["name"]: t for t in TOOLS}
ALL_TOOLS["get_sofa"] = _fn("get_sofa", "Current SOFA score and its six components.", {})
ALL_TOOLS["get_infection_data"] = _fn("get_infection_data", "Antibiotics and cultures ordered so far.", {})
MAIN_TOOLS, MAIN_SAMPLING = list(TOOLS), dict(SAMPLING)
def sampling_for(thinking):
    return (dict(temperature=0.6, top_p=0.95, extra={"top_k": 20}) if thinking
            else dict(temperature=0.7, top_p=0.8, extra={"top_k": 20}))

# ---- leakage test for the two extra tools ----
cur = con.cursor(); bad = 0
for r in eval_points.sample(min(30, len(eval_points)), random_state=2).itertuples():
    env = Env(cur, r.icustay_id, r.hour_time); env.get_sofa(); env.get_infection_data()
    bad += sum(t > env.now for t in env.seen)
assert bad == 0, "TIME-GATING VIOLATION in get_sofa / get_infection_data"
print("PASS: get_sofa and get_infection_data return no future timestamps")

# ---- single-prompt baseline: same data a typical agent fetches, but no choice of what to look up ----
SYSTEM_SINGLE = """You are an ICU clinical decision-support assistant. You are given data for one adult ICU patient at a given moment.
Task: estimate the probability that this patient will develop sepsis (suspected infection plus acute organ dysfunction) within the NEXT 12 HOURS.
Rules:
- Use only the data provided. Never guess or invent values.
- Final answer: ONLY a JSON object, no other text:
{"sepsis_risk": <number 0-1>, "alert": <true|false>, "evidence": ["short statement with the actual numbers and times", ...], "missing_data": ["...", ...]}
- Set alert=true only if the risk is high enough that a clinician should be notified now. Alerts that are wrong cause alert fatigue."""

def patient_header(env):
    return (f"Patient: {int(env.age)}-year-old {'man' if env.gender == 'M' else 'woman'}, admitted to {env.unit}. "
            f"ICU admission time: {_t(env.intime)}. Current time: {_t(env.now)} "
            f"({(env.now - pd.Timestamp(env.intime)).total_seconds() / 3600:.0f} hours into the ICU stay). "
            "Assess the risk of sepsis onset in the next 12 hours.")

def run_single(env, thinking=True):
    data = {"vitals_last_24h": env.get_vitals(24), "labs_last_48h": env.get_labs(list(LABS), 48),
            "urine_output_last_24h": env.get_urine_output(24)}
    for kw in ["infection", "fever", "hypotension", "antibiotic"]:
        data[f"notes_matching_{kw}"] = env.search_notes(kw, 3)
    blob = json.dumps(data, default=str); env.tool_log.append(blob)
    messages = [{"role": "system", "content": SYSTEM_SINGLE},
                {"role": "user", "content": patient_header(env) + "\n\nAvailable data (JSON):\n" + blob}]
    samp = sampling_for(thinking); ptok = ctok = n_trunc = 0; final, retried, trace, t0 = None, False, [], time.time()
    for attempt in range(2):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, max_tokens=MAX_TOKENS, temperature=samp["temperature"], top_p=samp["top_p"],
            seed=SEED + attempt, extra_body={"chat_template_kwargs": {"enable_thinking": thinking}, **samp["extra"]})
        ch = resp.choices[0]; ptok += resp.usage.prompt_tokens; ctok += resp.usage.completion_tokens
        n_trunc += ch.finish_reason == "length"
        trace.append({"step": attempt, "type": "final", "content": ch.message.content, "finish_reason": ch.finish_reason})
        final = parse_final(ch.message.content)
        if final is not None: break
        retried = True
        messages += [{"role": "assistant", "content": ch.message.content or ""},
                     {"role": "user", "content": "Return ONLY the final JSON object in the required format, nothing else."}]
    return {"final": final, "trace": trace, "n_tool_calls": 0, "prompt_tokens": ptok, "completion_tokens": ctok,
            "json_retry": retried, "n_truncated": int(n_trunc), "latency_s": round(time.time() - t0, 1), "tool_log": env.tool_log}
print("ablations defined:", list(ABLATIONS), "| will run:", RUN_ABLATIONS)

# ---- run the selected ablations (resumable; each writes abl_<name>.jsonl) ----
def read_jsonl(path):
    """Read a results file, skipping blank or half-written lines (e.g. from an interrupted run)."""
    rows, bad = [], 0
    if os.path.exists(path):
        for line in open(path):
            if not line.strip(): continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: bad += 1
    if bad: print(f"  note: skipped {bad} broken line(s) in {os.path.basename(path)}; those points will be re-run")
    return rows

def run_ablation(name):
    global TOOLS, SAMPLING
    cfg = ABLATIONS[name]
    path = f"{OUT_DIR}/abl_{name}.jsonl"
    pts = eval_points if not ABL_LIMIT else eval_points.sample(ABL_LIMIT, random_state=SEED)
    done = {(d["icustay_id"], d["hr"]) for d in read_jsonl(path)}
    todo = [r for r in pts.itertuples() if (int(r.icustay_id), int(r.hr)) not in done]
    print(f"\n=== {name}: {len(done)} done, {len(todo)} to run -> {path}", flush=True)
    if not todo: return
    TOOLS, SAMPLING = [ALL_TOOLS[n] for n in cfg["tools"]], sampling_for(cfg["thinking"])
    lock = threading.Lock()
    def work(r):
        try:
            env = Env(con.cursor(), r.icustay_id, r.hour_time)
            out = run_agent(env, thinking=cfg["thinking"]) if cfg["mode"] == "agent" else run_single(env, cfg["thinking"])
        except Exception as e:
            out = {"final": None, "trace": [{"type": "error", "content": f"{type(e).__name__}: {e}"}], "n_tool_calls": 0,
                   "prompt_tokens": 0, "completion_tokens": 0, "latency_s": 0, "tool_log": [], "json_retry": False, "n_truncated": 0}
        rec = {"icustay_id": int(r.icustay_id), "hr": int(r.hr), "y": int(r.y), "run": name, **out}
        with lock:
            with open(path, "a") as f: f.write(json.dumps(rec, default=str) + "\n")
        return rec
    try:
        t0, ok = time.time(), 0
        with ThreadPoolExecutor(CONCURRENCY) as ex:
            futs = [ex.submit(work, r) for r in todo]
            for i, f in enumerate(as_completed(futs), 1):
                ok += f.result()["final"] is not None
                if i % 25 == 0 or i == len(futs):
                    print(f"  {i}/{len(futs)} | valid {ok} | {(time.time() - t0) / 60:.1f} min", flush=True)
    finally:
        TOOLS, SAMPLING = MAIN_TOOLS, MAIN_SAMPLING     # restore the main configuration

for name in RUN_ABLATIONS:
    run_ablation(name)
print("\nall selected ablations finished")

def load(path):
    """Read a results file, skipping blank/half-written lines; keep one record per point (the latest)."""
    rows, bad = {}, 0
    for line in open(path):
        if not line.strip(): continue
        try: r = json.loads(line)
        except json.JSONDecodeError: bad += 1; continue
        k = (int(r["icustay_id"]), int(r["hr"]))
        if k in keep: rows[k] = r
    if bad: print(f"note: skipped {bad} broken line(s) in {os.path.basename(path)}")
    return list(rows.values())

# ---- Part 11. Ablation runs ----
from sklearn.metrics import roc_auc_score, average_precision_score
# ---- compare every run with the main run on the same decision points ----
num_re = re.compile(r"(?<![\w.])-?\d+(?:\.\d+)?")
ts_re = re.compile(r"\d{4}-\d{2}-\d{2}(?:[ T]\d{1,2}:\d{2}(?::\d{2})?)?|\b\d{1,2}:\d{2}(?::\d{2})?\b")
def _nums(s): return [float(x) for x in num_re.findall(ts_re.sub(" ", s))]
def faithfulness(rows):
    chk = sup = 0
    for r in rows:
        if not r["final"]: continue
        tn = np.array(sorted(set(n for t in r["tool_log"] for n in _nums(t))) or [np.nan])
        for e in r["final"].get("evidence", []):
            for n in _nums(str(e)):
                if abs(n) < 1 or n in (12.0, 24.0, 48.0): continue
                chk += 1; sup += bool(np.any(np.isclose(tn, n, rtol=0.01, atol=0.051)))
    return sup / max(chk, 1)

keep = set(zip(eval_points.icustay_id.astype(int), eval_points.hr.astype(int)))
def load(path):
    """Read a results file, skipping blank/half-written lines; keep one record per point (the latest)."""
    rows, bad = {}, 0
    for line in open(path):
        if not line.strip(): continue
        try: r = json.loads(line)
        except json.JSONDecodeError: bad += 1; continue
        k = (int(r["icustay_id"]), int(r["hr"]))
        if k in keep: rows[k] = r
    if bad: print(f"note: skipped {bad} broken line(s) in {os.path.basename(path)}")
    return list(rows.values())

runs = {"main": RES_PATH}
runs.update({n: f"{OUT_DIR}/abl_{n}.jsonl" for n in ABLATIONS if os.path.exists(f"{OUT_DIR}/abl_{n}.jsonl")})
base = dv[["icustay_id", "hr", "y", "sofa", "gb"]].copy()
for n, p in runs.items():
    rows_n = load(p)
    d = pd.DataFrame([{"icustay_id": r["icustay_id"], "hr": r["hr"],
                       f"risk_{n}": r["final"]["sepsis_risk"] if r["final"] else np.nan,
                       f"alert_{n}": float(r["final"]["alert"]) if r["final"] else np.nan} for r in rows_n])
    base = base.merge(d, on=["icustay_id", "hr"], how="left")
    runs[n] = (p, rows_n)

N_BOOT = globals().get("N_BOOT", 2000)
# stay-level bootstrap, paired with the main run on points where both answered
dvb = base.reset_index(drop=True)
groups = list(dvb.groupby("icustay_id").indices.values())
rng_a = np.random.default_rng(SEED)
boot_ix = [np.concatenate([groups[i] for i in rng_a.integers(0, len(groups), len(groups))]) for _ in range(N_BOOT)]

def auroc_ci(mask, col, ref=None):
    y = dvb.y.values; s = dvb[col].values; r_ = dvb[ref].values if ref else None
    pt = roc_auc_score(y[mask], s[mask]); diffs, vals = [], []
    for ix in boot_ix:
        ix = ix[mask[ix]]; yb = y[ix]
        if yb.min() == yb.max(): continue
        a = roc_auc_score(yb, s[ix]); vals.append(a)
        if ref: diffs.append(a - roc_auc_score(yb, r_[ix]))
    out = {"AUROC (95% CI)": f"{pt:.3f} ({np.percentile(vals, 2.5):.3f}-{np.percentile(vals, 97.5):.3f})"}
    if ref:
        d = np.array(diffs); p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
        out["vs main (95% CI)"] = (f"{pt - roc_auc_score(y[mask], r_[mask]):+.3f} "
                                   f"({np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f})")
        out["p"] = "<0.001" if p < 1e-3 else round(p, 3)
    return out

summary = []
for n, (p, rows_n) in runs.items():
    m = dvb[f"risk_{n}"].notna().values & dvb["risk_main"].notna().values
    a = dvb.loc[m, f"alert_{n}"].astype(int); yy = dvb.loc[m, "y"].astype(int)
    rr = pd.DataFrame(rows_n)
    row = {"run": n, "n": int(m.sum()), "n missing": int(len(dvb) - len(rows_n)),
           "valid %": round(100 * dvb[f"risk_{n}"].notna().mean(), 1),
           **auroc_ci(m, f"risk_{n}", None if n == "main" else "risk_main"),
           "AUPRC": round(average_precision_score(yy, dvb.loc[m, f"risk_{n}"]), 3),
           "alert rate": round(a.mean(), 3),
           "sens": round(((a == 1) & (yy == 1)).sum() / max((yy == 1).sum(), 1), 3),
           "spec": round(((a == 0) & (yy == 0)).sum() / max((yy == 0).sum(), 1), 3),
           "tool calls": round(rr.n_tool_calls.mean(), 1),
           "tokens": int((rr.prompt_tokens + rr.completion_tokens).mean()),
           "latency s": round(rr.latency_s.mean(), 1),
           "faithful %": round(100 * faithfulness(rows_n), 1)}
    summary.append(row)
abl_table = pd.DataFrame(summary).set_index("run")
pd.set_option("display.width", 250)
print("SOFA AUROC on the same points:", round(roc_auc_score(dvb.y, dvb.sofa), 3),
      "| gradient boosting:", round(roc_auc_score(dvb.y, dvb.gb), 3))
print(abl_table.to_string())
abl_table.to_csv(f"{OUT_DIR}/ablation_summary.csv")

# which tools did each agent run actually use?
use = {}
for n, (p, rows_n) in runs.items():
    c = pd.Series([t["name"] for r in rows_n for t in r["trace"] if t.get("type") == "tool"], dtype=object).value_counts()
    use[n] = (c / max(len(rows_n), 1)).round(2)
print("\nMean calls per case, by tool:")
print(pd.DataFrame(use).fillna(0).to_string())
print("saved", f"{OUT_DIR}/ablation_summary.csv")


In [ ]:
# =====================================================================================
# PART 12A. Extended evaluation: features, classic scores and gradient-boosting comparators
# Time-gated feature table, shared helpers (bootstrap, weights, calibration, net benefit),
# SIRS/qSOFA/NEWS2/delta-SOFA, tuned + Platt-calibrated GB on three feature sets,
# native-prevalence out-of-fold evaluation on all decision points.
# =====================================================================================
# ---- Part 12 configuration ----
# ===== Part 12 configuration =====
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

FEAT_PATH         = f"{OUT_DIR}/dp_features.parquet"     # features at every decision-grid hour of every cohort stay
REBUILD_FEATURES  = False
TUNE_GB           = True      # small hyperparameter grid with grouped CV (about 20-40 min on the Spark); False = defaults
REUSE_GB_PARAMS   = True      # True: reuse the tuned parameters saved in gb_params.joblib (same result, skips the tuning after a restart)
N_BOOT            = globals().get("N_BOOT", 2000)        # bootstrap resamples on the 568-point sample
N_BOOT_FULL       = 200       # bootstrap resamples on all ~400k decision points (each resample is slow)
ALERTS_PER_DAY    = 24 / DECISION_EVERY_H                # decision points per patient-day
VIT_VARS = ["heart_rate", "sbp", "map", "resp_rate", "temp_c", "spo2"]
LABS_X = dict(LABS); LABS_X["paco2"] = 50818              # PaCO2 is needed for SIRS
if "RES_PATH" not in globals():          # defined in Part 8; recreated here so Part 8 need not be re-run after a restart
    RUN_NAME = (f"agent_{'think' if THINKING else 'nothink'}_{'noinf' if HIDE_INFECTION_DATA else 'inf'}"
                f"_{'sofa' if ENABLE_SOFA_TOOL else 'nosofa'}_gcs-{INTUBATED_GCS}")
    RES_PATH = f"{OUT_DIR}/{RUN_NAME}.jsonl"
print("Part 12 configured | features:", FEAT_PATH, "| exists:", os.path.exists(FEAT_PATH))

# ---- 12a, feature table ----
def build_features():
    """Writes FEAT_PATH. Scratch tables go to WORK_DB (this notebook's own file); MIMIC-III is only read."""
    wk = con
    if True:
        t0 = time.time()
        # ---- vital signs, placed at the hour they became visible ----
        vit_ids = [i for ids in VITALS.values() for i in ids]
        vit_case = "CASE " + " ".join(f"WHEN ce.itemid IN ({', '.join(map(str, ids))}) THEN '{k}'" for k, ids in VITALS.items()) + " END"
        tf = ", ".join(map(str, VITALS["temp_f"]))
        wk.execute(f"""
        CREATE OR REPLACE TABLE fv AS
        WITH x AS (
          SELECT ce.icustay_id, ce.charttime,
                 {HR('GREATEST(ce.charttime, COALESCE(ce.storetime, ce.charttime))')} AS hr,
                 {vit_case} AS var,
                 CASE WHEN ce.itemid IN ({tf}) THEN (ce.valuenum - 32) * 5.0 / 9 ELSE ce.valuenum END AS v
          FROM chartevents ce JOIN cohort_final c ON c.icustay_id = ce.icustay_id
          WHERE ce.itemid IN ({', '.join(map(str, vit_ids))}) AND ce.valuenum IS NOT NULL
            AND (ce.error IS NULL OR ce.error = 0)
            AND ce.charttime >= c.intime - INTERVAL 24 HOUR AND ce.charttime <= c.outtime)
        SELECT icustay_id, hr, charttime, CASE WHEN var = 'temp_f' THEN 'temp_c' ELSE var END AS var, v FROM x
        WHERE (var IN ('heart_rate', 'sbp', 'map') AND v > 0 AND v < 300)
           OR (var = 'resp_rate' AND v > 0 AND v < 80)
           OR (var IN ('temp_c', 'temp_f') AND v > 25 AND v < 45)
           OR (var = 'spo2' AND v > 0 AND v <= 100)""")
        agg = []
        for v in VIT_VARS:
            f = f"FILTER (WHERE var = '{v}')"
            agg += [f"SUM(v) {f} AS {v}_sum", f"COUNT(v) {f} AS {v}_n", f"MIN(v) {f} AS {v}_mn",
                    f"MAX(v) {f} AS {v}_mx", f"max_by(v, charttime) {f} AS {v}_lst"]
        wk.execute(f"CREATE OR REPLACE TABLE fv_h AS SELECT icustay_id, hr, {', '.join(agg)} FROM fv GROUP BY 1, 2")
        print(f"  vitals done ({(time.time()-t0)/60:.1f} min)", flush=True)

        # ---- labs (no storetime in MIMIC-III; placed at charttime) ----
        lab_case = "CASE " + " ".join(f"WHEN l.itemid = {i} THEN '{k}'" for k, i in LABS_X.items()) + " END"
        wk.execute(f"""
        CREATE OR REPLACE TABLE fl AS
        SELECT c.icustay_id, {HR('l.charttime')} AS hr, l.charttime, {lab_case} AS var, l.valuenum AS v
        FROM labevents l JOIN cohort_final c ON c.hadm_id = l.hadm_id
        WHERE l.itemid IN ({', '.join(map(str, LABS_X.values()))}) AND l.valuenum IS NOT NULL AND l.valuenum >= 0
          AND l.charttime >= c.intime - INTERVAL 48 HOUR AND l.charttime <= c.outtime""")
        agg = ["COUNT(*) AS lab_n"]
        for k in LABS_X:
            f = f"FILTER (WHERE var = '{k}')"
            agg += [f"MIN(v) {f} AS {k}_mn", f"MAX(v) {f} AS {k}_mx", f"max_by(v, charttime) {f} AS {k}_lst",
                    f"COUNT(v) {f} AS {k}_cnt"]
        wk.execute(f"CREATE OR REPLACE TABLE fl_h AS SELECT icustay_id, hr, {', '.join(agg)} FROM fl GROUP BY 1, 2")
        wk.execute(f"""CREATE OR REPLACE TABLE fo_h AS
                       SELECT f.icustay_id, {HR('f.charttime')} AS hr, MAX(f.fio2) AS fio2
                       FROM ev_fio2 f JOIN cohort_final c ON c.icustay_id = f.icustay_id GROUP BY 1, 2""")
        print(f"  labs done ({(time.time()-t0)/60:.1f} min)", flush=True)

        # ---- rolling windows over the dense hourly grid ----
        vcols = [f"{v}_{s}" for v in VIT_VARS for s in ("sum", "n", "mn", "mx", "lst")]
        lcols = ["lab_n"] + [f"{k}_{s}" for k in LABS_X for s in ("mn", "mx", "lst", "cnt")]
        win = []
        for v in VIT_VARS:
            win += [f"LAST_VALUE({v}_lst IGNORE NULLS) OVER w24 AS {v}_last24",
                    f"MIN({v}_mn) OVER w24 AS {v}_min24", f"MAX({v}_mx) OVER w24 AS {v}_max24",
                    f"SUM({v}_sum) OVER w24 / NULLIF(SUM({v}_n) OVER w24, 0) AS {v}_mean24",
                    f"SUM({v}_sum) OVER w6 / NULLIF(SUM({v}_n) OVER w6, 0) AS {v}_mean6",
                    f"SUM({v}_sum) OVER wprev / NULLIF(SUM({v}_n) OVER wprev, 0) AS {v}_meanprev",
                    f"COALESCE(SUM({v}_n) OVER w24, 0) AS {v}_n24"]
        for k in LABS_X:
            win += [f"LAST_VALUE({k}_lst IGNORE NULLS) OVER w48 AS {k}_last",
                    f"MIN({k}_mn) OVER w24 AS {k}_min24", f"MAX({k}_mx) OVER w24 AS {k}_max24",
                    f"MIN({k}_mn) OVER w48 AS {k}_min48", f"COALESCE(SUM({k}_cnt) OVER w24, 0) AS {k}_n24"]
        win += ["COALESCE(SUM(lab_n) OVER w24, 0) AS lab_n24",
                "SUM(urine_ml) OVER w6 AS urine_6h", "SUM(urine_ml) OVER w24 AS urine_24h_any",
                "COUNT(urine_ml) OVER w6 AS urine_n6", "MAX(fio2) OVER w24 AS fio2_max24"]
        wk.execute(f"""
        CREATE OR REPLACE TABLE feat_grid AS
        WITH j AS (
          SELECT g.icustay_id, g.hr, {', '.join('v.' + c for c in vcols)}, {', '.join('l.' + c for c in lcols)},
                 u.ml AS urine_ml, o.fio2
          FROM hourly_grid g
          LEFT JOIN fv_h v ON v.icustay_id = g.icustay_id AND v.hr = g.hr
          LEFT JOIN fl_h l ON l.icustay_id = g.icustay_id AND l.hr = g.hr
          LEFT JOIN ev_urine u ON u.icustay_id = g.icustay_id AND u.hr = g.hr
          LEFT JOIN fo_h o ON o.icustay_id = g.icustay_id AND o.hr = g.hr)
        SELECT icustay_id, hr, {', '.join(win)}
        FROM j
        WINDOW w6    AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 5 PRECEDING AND CURRENT ROW),
               w24   AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW),
               wprev AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND 18 PRECEDING),
               w48   AS (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 47 PRECEDING AND CURRENT ROW)""")
        print(f"  rolling windows done ({(time.time()-t0)/60:.1f} min)", flush=True)

        # ---- decision-grid hours + SOFA history + static variables ----
        wk.execute(f"""
        CREATE OR REPLACE TABLE dp_features AS
        WITH sh AS (
          SELECT s.*,
                 LAG(s.sofa, 24) OVER (PARTITION BY s.icustay_id ORDER BY s.hr) AS sofa_lag24,
                 MIN(s.sofa) OVER (PARTITION BY s.icustay_id ORDER BY s.hr ROWS BETWEEN 72 PRECEDING AND 1 PRECEDING) AS sofa_min72prev
          FROM sofa_hourly s)
        SELECT f.*, sh.hour_time, sh.sofa, sh.s_resp, sh.s_coag, sh.s_liver, sh.s_cardio, sh.s_cns, sh.s_renal,
               sh.platelets_min, sh.bilirubin_max, sh.creatinine_max, sh.map_min, sh.gcs_min, sh.urine_24h, sh.pf_min,
               sh.vent, sh.dopa, sh.dobu, sh.epi, sh.norepi, sh.phenyl, sh.vasop, sh.sofa_lag24, sh.sofa_min72prev,
               c.hadm_id, c.age, c.gender, c.first_careunit, c.intime, c.outtime, c.hospital_expire_flag,
               i.dbsource, a.admittime, COALESCE(w.weight_kg, {DEFAULT_WEIGHT_KG}) AS weight_kg,
               date_diff('second', a.admittime, c.intime) / 3600.0 + f.hr AS hosp_hr
        FROM feat_grid f
        JOIN sh ON sh.icustay_id = f.icustay_id AND sh.hr = f.hr
        JOIN cohort_final c ON c.icustay_id = f.icustay_id
        JOIN icustays i ON i.icustay_id = f.icustay_id
        JOIN admissions a ON a.hadm_id = c.hadm_id
        LEFT JOIN stay_weight w ON w.icustay_id = f.icustay_id
        WHERE f.hr >= {MIN_ONSET_HR} AND f.hr % {DECISION_EVERY_H} = 0""")
        d = wk.execute("SELECT * FROM dp_features").df()
        d.columns = d.columns.str.lower()
        d.to_parquet(FEAT_PATH, index=False)
        print(f"features: {len(d):,} rows x {d.shape[1]} columns from {d.icustay_id.nunique():,} stays "
              f"({(time.time()-t0)/60:.1f} min) -> {FEAT_PATH}")
        for t in ["fv", "fv_h", "fl", "fl_h", "fo_h", "feat_grid", "dp_features"]:
            wk.execute(f"DROP TABLE IF EXISTS {t}")      # free disk space; the parquet file keeps the result

if REBUILD_FEATURES or not os.path.exists(FEAT_PATH):
    build_features()

FEAT = pd.read_parquet(FEAT_PATH)
for c in ["hour_time", "intime", "outtime", "admittime"]:
    FEAT[c] = pd.to_datetime(FEAT[c])
# derived features
for v in VIT_VARS:
    FEAT[f"{v}_trend"] = FEAT[f"{v}_mean6"] - FEAT[f"{v}_meanprev"]
FEAT["creatinine_delta48"] = FEAT["creatinine_last"] - FEAT["creatinine_min48"]
FEAT["urine_6h_mlkgh"]  = FEAT["urine_6h"] / FEAT["weight_kg"] / 6
FEAT["urine_24h_mlkgh"] = FEAT["urine_24h_any"] / FEAT["weight_kg"] / 24
FEAT["male"] = (FEAT["gender"] == "M").astype(int)
for u in ["MICU", "SICU", "CCU", "CSRU", "TSICU"]:
    FEAT[f"unit_{u.lower()}"] = (FEAT["first_careunit"] == u).astype(int)
FEAT["dsofa24"] = (FEAT["sofa"] - FEAT["sofa_lag24"]).fillna(0)
FEAT["dsofa72"] = (FEAT["sofa"] - FEAT["sofa_min72prev"]).fillna(0)
FEAT["era"] = FEAT["dbsource"].map({"carevue": "CareVue", "metavision": "MetaVision"}).fillna("both")

# QC: features must be missing at roughly the same rate as the agent would see the data
qc = FEAT[["heart_rate_last24", "resp_rate_last24", "temp_c_last24", "lactate_last", "wbc_last", "paco2_min24", "urine_6h"]].notna().mean().mul(100).round(1)
print("share of decision-grid hours with a value (%):"); print(qc.to_string())
assert FEAT[["icustay_id", "hr"]].duplicated().sum() == 0, "duplicate (stay, hour) rows in the feature table"

# ---- 12b, helpers ----
def read_results(path, keys=None):
    """Read an agent results JSONL; skip blank/broken lines; keep the latest record per (stay, hour)."""
    rows = {}
    if not os.path.exists(path): return []
    for line in open(path):
        if not line.strip(): continue
        try: r = json.loads(line)
        except json.JSONDecodeError: continue
        k = (int(r["icustay_id"]), int(r["hr"]))
        if keys is None or k in keys: rows[k] = r
    return list(rows.values())

def results_frame(rows, tag):
    return pd.DataFrame([{"icustay_id": int(r["icustay_id"]), "hr": int(r["hr"]),
                          f"risk_{tag}": float(r["final"]["sepsis_risk"]) if r["final"] else np.nan,
                          f"alert_{tag}": float(bool(r["final"]["alert"])) if r["final"] else np.nan} for r in rows])

def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def expit(x): return 1 / (1 + np.exp(-np.asarray(x, dtype=float)))

def boot_groups(stays):
    return list(pd.DataFrame({"s": np.asarray(stays)}).groupby("s").indices.values())

def boot_indices(stays, n_boot, seed=SEED):
    g = boot_groups(stays); rng = np.random.default_rng(seed)
    return [np.concatenate([g[i] for i in rng.integers(0, len(g), len(g))]) for _ in range(n_boot)]

def wauc(y, s, w=None):
    try: return roc_auc_score(y, s, sample_weight=w)
    except ValueError: return np.nan

def wap(y, s, w=None):
    try: return average_precision_score(y, s, sample_weight=w)
    except ValueError: return np.nan

def boot_table(y, scores, stays, w=None, ref=None, n_boot=N_BOOT, metrics=("AUROC", "AUPRC"), seed=SEED, verbose=False):
    """Point estimate (95% CI) for every score; paired difference vs `ref` on the same stay-level resamples."""
    y = np.asarray(y).astype(int); w = None if w is None else np.asarray(w, dtype=float)
    fn = {"AUROC": wauc, "AUPRC": wap}
    idx = boot_indices(stays, n_boot, seed)
    out = {}; t_start = time.time()
    for k_, (name, s) in enumerate(scores.items()):
        if verbose:
            print(f"  bootstrap {k_+1}/{len(scores)}: {name} ({(time.time()-t_start)/60:.1f} min elapsed)", flush=True)
        s = np.asarray(s, dtype=float); row = {}
        for m in metrics:
            pt = fn[m](y, s, w); vals, diffs = [], []
            for ix in idx:
                yb = y[ix]
                if yb.min() == yb.max(): continue
                wb = None if w is None else w[ix]
                a = fn[m](yb, s[ix], wb); vals.append(a)
                if ref is not None and name != ref:
                    diffs.append(a - fn[m](yb, np.asarray(scores[ref], dtype=float)[ix], wb))
            row[m] = f"{pt:.3f} ({np.nanpercentile(vals, 2.5):.3f}-{np.nanpercentile(vals, 97.5):.3f})"
            if diffs:
                d = np.array(diffs); p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
                ptd = pt - fn[m](y, np.asarray(scores[ref], dtype=float), w)
                row[f"Δ{m} vs {ref} (95% CI)"] = f"{ptd:+.3f} ({np.percentile(d, 2.5):+.3f} to {np.percentile(d, 97.5):+.3f})"
                row[f"p ({m})"] = "<0.001" if p < 1e-3 else round(p, 3)
        out[name] = row
    return pd.DataFrame(out).T

def sample_weights(points, dp_all):
    """Inverse-probability weights that map the stratified evaluation sample back to all eligible decision points.
    weight = (eligible stays in class / sampled stays in class) x (eligible points in stay / sampled points in stay)."""
    st = dp_all.groupby("icustay_id").agg(stay_class=("stay_class", "first"), n_pts=("hr", "size"))
    n_cls = st.stay_class.value_counts()
    sp = points.groupby("icustay_id").agg(stay_class=("stay_class", "first"), n_samp=("hr", "size")).join(st["n_pts"])
    s_cls = sp.stay_class.value_counts()
    sp["w"] = sp.stay_class.map(n_cls / s_cls) * sp.n_pts / sp.n_samp
    return points.icustay_id.map(sp.w).values.astype(float)

def citl(y, lp, w):
    """Calibration-in-the-large: intercept a with slope fixed at 1 (Newton's method)."""
    a = 0.0
    for _ in range(50):
        p = expit(a + lp); g = np.sum(w * (y - p)); h = np.sum(w * p * (1 - p))
        if h <= 0: break
        a += g / h
        if abs(g / h) < 1e-8: break
    return a

def calib_stats(y, p, w=None):
    y = np.asarray(y).astype(int); p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    w = np.ones_like(p) if w is None else np.asarray(w, dtype=float)
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=2000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    return {"Brier": round(float(np.average((p - y) ** 2, weights=w)), 4),
            "O/E": round(float(np.sum(w * y) / np.sum(w * p)), 2),
            "CITL": round(float(citl(y, lp, w)), 2), "slope": round(float(slope), 2),
            "mean predicted": round(float(np.average(p, weights=w)), 4), "observed": round(float(np.average(y, weights=w)), 4)}

def crossfit_platt(x, y, groups, w=None, k=5):
    """Platt scaling fitted out-of-fold by stay, so every calibrated value is out-of-sample."""
    x = np.asarray(x, dtype=float).reshape(-1, 1); y = np.asarray(y).astype(int); out = np.full(len(y), np.nan)
    for tr, te in GroupKFold(n_splits=k).split(x, y, groups):
        lr = LogisticRegression(C=1e6, max_iter=2000).fit(x[tr], y[tr], sample_weight=None if w is None else np.asarray(w)[tr])
        out[te] = lr.predict_proba(x[te])[:, 1]
    return out

def op_point(y, a, w=None):
    y = np.asarray(y).astype(int); a = np.asarray(a).astype(bool); w = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = np.sum(w * (a & (y == 1))); fp = np.sum(w * (a & (y == 0))); fn = np.sum(w * (~a & (y == 1))); tn = np.sum(w * (~a & (y == 0)))
    rate = (tp + fp) / w.sum(); ppv = tp / max(tp + fp, 1e-12)
    return {"alert rate": round(rate, 3), "sensitivity": round(tp / max(tp + fn, 1e-12), 3),
            "specificity": round(tn / max(tn + fp, 1e-12), 3), "PPV": round(ppv, 3),
            "alerts per patient-day": round(rate * ALERTS_PER_DAY, 2),
            "alerts per true alert (1/PPV)": round(1 / ppv, 1) if ppv > 0 else np.inf}

def thr_for_rate(p, rate, w=None):
    """Threshold whose (weighted) alert rate equals `rate`."""
    p = np.asarray(p, dtype=float); w = np.ones(len(p)) if w is None else np.asarray(w, dtype=float)
    o = np.argsort(-p); cw = np.cumsum(w[o]) / w.sum()
    i = min(np.searchsorted(cw, rate), len(p) - 1)
    return p[o][i]

def net_benefit(y, p, w, thresholds):
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); w = np.asarray(w, dtype=float); n = w.sum()
    out = []
    for t in thresholds:
        a = p >= t
        tp = np.sum(w * (a & (y == 1))) / n; fp = np.sum(w * (a & (y == 0))) / n
        out.append(tp - fp * t / (1 - t))
    return np.array(out)

print("helpers ready")

# ---- 12c, classic scores ----
def add_scores(d):
    d = d.copy()
    d["sirs"] = (((d.temp_c_max24 > 38) | (d.temp_c_min24 < 36)).astype(int) + (d.heart_rate_max24 > 90).astype(int)
                 + ((d.resp_rate_max24 > 20) | (d.paco2_min24 < 32)).astype(int)
                 + ((d.wbc_max24 > 12) | (d.wbc_min24 < 4)).astype(int))
    d["qsofa"] = (d.resp_rate_max24 >= 22).astype(int) + (d.sbp_min24 <= 100).astype(int) + (d.gcs_min < 15).astype(int)
    rr, sp, sb, hrt, tc = d.resp_rate_last24, d.spo2_last24, d.sbp_last24, d.heart_rate_last24, d.temp_c_last24
    news = (np.select([rr <= 8, rr <= 11, rr <= 20, rr <= 24, rr >= 25], [3, 1, 0, 2, 3], 0)
            + np.select([sp <= 91, sp <= 93, sp <= 95], [3, 2, 1], 0)
            + np.where((d.fio2_max24 > 0.21) | (d.vent == 1), 2, 0)
            + np.select([sb <= 90, sb <= 100, sb <= 110, sb >= 220], [3, 2, 1, 3], 0)
            + np.select([hrt <= 40, hrt <= 50, hrt <= 90, hrt <= 110, hrt <= 130, hrt > 130], [3, 1, 0, 1, 2, 3], 0)
            + np.select([tc <= 35, tc <= 36, tc <= 38, tc <= 39, tc > 39], [3, 1, 0, 1, 2], 0)
            + np.where(d.gcs_min < 15, 3, 0))
    d["news2"] = news
    return d

FEAT = add_scores(FEAT)
SCORES = ["sofa", "dsofa24", "dsofa72", "sirs", "qsofa", "news2"]
SCORE_NAMES = {"sofa": "SOFA total", "dsofa24": "ΔSOFA (24 h)", "dsofa72": "ΔSOFA (vs 72 h minimum)",
               "sirs": "SIRS", "qsofa": "qSOFA", "news2": "NEWS2"}

# main-label decision points (exactly the Part 5 table) with features attached
DP = con.execute("SELECT icustay_id, hr, y, stay_class, onset_hr FROM decision_points").df()
DP.columns = DP.columns.str.lower()
DPF = DP.merge(FEAT, on=["icustay_id", "hr"], how="left")
assert DPF.sofa.notna().all(), "some decision points have no feature row"
print(f"{len(DPF):,} decision points, {DPF.y.mean()*100:.2f}% positive")
print(DPF[SCORES].describe().T[["mean", "50%", "max"]].round(2).to_string())

# ---- 12d, GB models ----
VIT_F = [f"{v}_{s}" for v in VIT_VARS for s in ("last24", "min24", "max24", "mean24", "mean6", "trend", "n24")]
LAB_F = ([f"{k}_last" for k in LABS_X]
         + [f"{k}_{s}" for k in ("lactate", "wbc", "creatinine", "platelets", "bilirubin", "bicarbonate", "ph", "paco2", "glucose") for s in ("min24", "max24")]
         + ["creatinine_delta48", "lactate_n24", "lab_n24"])
URINE_F = ["urine_6h_mlkgh", "urine_24h_mlkgh", "urine_n6"]
STATIC_F = ["age", "male", "unit_micu", "unit_sicu", "unit_ccu", "unit_csru", "unit_tsicu", "hr", "hosp_hr"]
SOFA_F = ["sofa", "s_resp", "s_coag", "s_liver", "s_cardio", "s_cns", "s_renal", "platelets_min", "bilirubin_max",
          "creatinine_max", "map_min", "gcs_min", "urine_24h", "pf_min", "vent", "dopa", "dobu", "epi", "norepi",
          "phenyl", "vasop", "fio2_max24", "dsofa24", "dsofa72"]
FEATSETS = {"gb_sofa": FEATS,                                    # FEATS from Part 9
            "gb_matched": VIT_F + LAB_F + URINE_F + STATIC_F,
            "gb_full": VIT_F + LAB_F + URINE_F + STATIC_F + SOFA_F}
GB_NAMES = {"gb_sofa": "GB, SOFA inputs (Part 9)", "gb_matched": "GB, agent-matched", "gb_full": "GB, full"}
GB_GRID = [dict(learning_rate=lr, max_leaf_nodes=ml, min_samples_leaf=msl, l2_regularization=l2)
           for lr in (0.05, 0.1) for ml in (15, 31) for msl in (100,) for l2 in (0.0, 1.0)]
GB_DEFAULT = dict(learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=0.0)

def make_gb(params):
    return HistGradientBoostingClassifier(max_iter=500, early_stopping=True, validation_fraction=0.1,
                                          n_iter_no_change=30, random_state=SEED, **params)

def tune_gb(d, feats, max_stays=6000):
    """Grouped 3-fold CV on a random subset of training stays; selects by mean AUPRC."""
    if not TUNE_GB: return GB_DEFAULT
    rng = np.random.default_rng(SEED)
    st = d.icustay_id.unique(); st = rng.choice(st, min(max_stays, len(st)), replace=False)
    s = d[d.icustay_id.isin(set(st))].sort_values(["icustay_id", "hr"])
    best, best_score = GB_DEFAULT, -1
    for prm in GB_GRID:
        sc = []
        for tr, te in GroupKFold(3).split(s, s.y, s.icustay_id):
            m = make_gb(prm).fit(s.iloc[tr][feats], s.iloc[tr].y)
            sc.append(average_precision_score(s.iloc[te].y, m.predict_proba(s.iloc[te][feats])[:, 1]))
        if np.mean(sc) > best_score: best, best_score = prm, np.mean(sc)
    print(f"    best {best} (CV AUPRC {best_score:.3f})")
    return best

def fit_calibrated(train, feats, params, k=5):
    """Model on all training rows + Platt calibrator fitted on grouped out-of-fold predictions."""
    train = train.sort_values(["icustay_id", "hr"]).reset_index(drop=True)
    oof = np.zeros(len(train))
    for tr, te in GroupKFold(k).split(train, train.y, train.icustay_id):
        oof[te] = make_gb(params).fit(train.loc[tr, feats], train.loc[tr, "y"]).predict_proba(train.loc[te, feats])[:, 1]
    platt = LogisticRegression(C=1e6, max_iter=2000).fit(logit(oof).reshape(-1, 1), train.y)
    model = make_gb(params).fit(train[feats], train.y)
    def predict(X):
        return platt.predict_proba(logit(model.predict_proba(X[feats])[:, 1]).reshape(-1, 1))[:, 1]
    predict.model, predict.platt, predict.params, predict.feats = model, platt, params, feats
    return predict

# ---- evaluation sample = the 568 points of Part 9 ----
EVAL_STAYS = set(eval_points.icustay_id.astype(int))
TRAIN = DPF[~DPF.icustay_id.isin(EVAL_STAYS)]
import joblib
GB_PARAMS, GB_EVAL = {}, {}
_saved = {}
_pp = f"{OUT_DIR}/gb_params.joblib"
if REUSE_GB_PARAMS and TUNE_GB and os.path.exists(_pp):
    _j = joblib.load(_pp)
    if _j.get("featsets") == FEATSETS:
        _saved = _j["params"]; print("reusing tuned parameters from gb_params.joblib (set REUSE_GB_PARAMS = False to re-tune)")
for name, feats in FEATSETS.items():
    t0 = time.time(); print(f"{name}: {len(feats)} features", flush=True)
    GB_PARAMS[name] = _saved[name] if name in _saved else tune_gb(TRAIN, feats)
    GB_EVAL[name] = fit_calibrated(TRAIN, feats, GB_PARAMS[name])
    print(f"    fitted in {(time.time()-t0)/60:.1f} min")
import joblib
joblib.dump({"params": GB_PARAMS, "featsets": FEATSETS}, f"{OUT_DIR}/gb_params.joblib")
joblib.dump(GB_EVAL["gb_sofa"].model, f"{OUT_DIR}/gb_sofa_model.joblib")       # saved for reuse
joblib.dump(GB_EVAL["gb_sofa"].platt, f"{OUT_DIR}/gb_sofa_platt.joblib")

# ---- 12d, all-points predictions ----
# ---- native prevalence: grouped 5-fold OOF predictions for ALL decision points ----
def oof_all(d, feats, params, k=5):
    d = d.sort_values(["icustay_id", "hr"]).reset_index(drop=True)
    raw = np.zeros(len(d)); fold = np.zeros(len(d), dtype=int)
    for j, (tr, te) in enumerate(GroupKFold(k).split(d, d.y, d.icustay_id)):
        raw[te] = make_gb(params).fit(d.loc[tr, feats], d.loc[tr, "y"]).predict_proba(d.loc[te, feats])[:, 1]; fold[te] = j
    cal = np.zeros(len(d))
    for j in range(k):          # cross-fitted Platt: calibrator for fold j is fitted on the other folds' OOF scores
        o = fold != j
        lr = LogisticRegression(C=1e6, max_iter=2000).fit(logit(raw[o]).reshape(-1, 1), d.y[o])
        cal[fold == j] = lr.predict_proba(logit(raw[fold == j]).reshape(-1, 1))[:, 1]
    return d[["icustay_id", "hr"]].assign(**{"raw": raw, "cal": cal})

ALLP_PATH = f"{OUT_DIR}/allpoints_oof.parquet"
REUSE_OOF = True        # True: reuse allpoints_oof.parquet if it already holds every GB column (saves ~30-60 min)
_need = list(FEATSETS) + [f"{g}_raw" for g in FEATSETS]
if REUSE_OOF and os.path.exists(ALLP_PATH) and set(_need) <= set(pd.read_parquet(ALLP_PATH).columns) \
        and len(pd.read_parquet(ALLP_PATH)) == len(DPF):
    ALLP = pd.read_parquet(ALLP_PATH)
    print(f"reused {ALLP_PATH} ({len(ALLP):,} rows; set REUSE_OOF = False to refit)")
else:
    ALLP = DPF[["icustay_id", "hr", "y", "era"] + SCORES].copy()
    for name, feats in FEATSETS.items():
        t0 = time.time(); print(f"{name}: fitting 5 grouped folds on {len(DPF):,} points ...", flush=True)
        o = oof_all(DPF, feats, GB_PARAMS[name])
        ALLP = ALLP.merge(o.rename(columns={"raw": f"{name}_raw", "cal": name}), on=["icustay_id", "hr"])
        print(f"{name} OOF done ({(time.time()-t0)/60:.1f} min)", flush=True)
    ALLP.to_parquet(ALLP_PATH, index=False)
    print(f"saved {ALLP_PATH}")

# ---- native-prevalence discrimination table (stay-level bootstrap; prints progress per score) ----
# Runs on the full set of decision points, so 200 resamples can take 20-60 min; lower N_BOOT_FULL for a quick look.
if "ALLP" not in globals():
    ALLP = pd.read_parquet(f"{OUT_DIR}/allpoints_oof.parquet")
full_scores = {SCORE_NAMES[s]: ALLP[s].values for s in SCORES} | {GB_NAMES[g]: ALLP[g].values for g in FEATSETS}
native = boot_table(ALLP.y, full_scores, ALLP.icustay_id, ref=GB_NAMES["gb_sofa"], n_boot=N_BOOT_FULL, verbose=True)
native["calibration (GB only)"] = ""
for g in FEATSETS:
    native.loc[GB_NAMES[g], "calibration (GB only)"] = str(calib_stats(ALLP.y, ALLP[g]))
print(f"ALL {len(ALLP):,} decision points, native prevalence {ALLP.y.mean()*100:.2f}% (chance AUPRC = {ALLP.y.mean():.3f}); "
      f"{N_BOOT_FULL} stay-level bootstrap resamples")
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 200)
print(native.to_string())
native.to_csv(f"{OUT_DIR}/ext_native_prevalence.csv"); print("saved ext_native_prevalence.csv")


In [ ]:
# =====================================================================================
# PART 12B. Extended evaluation: weighted comparison, decision curves, stacking, onset drivers
# 568-point comparison reweighted to the true prevalence, calibration and operating
# points, decision-curve analysis, stacked incremental value, onset-time drivers.
# =====================================================================================
# ---- 12e, 568-point table ----
main_rows = read_results(RES_PATH, set(zip(eval_points.icustay_id.astype(int), eval_points.hr.astype(int))))
E = (eval_points[["icustay_id", "hr", "y", "stay_class", "onset_hr"]].astype({"icustay_id": int, "hr": int})
     .merge(results_frame(main_rows, "main"), on=["icustay_id", "hr"], how="left")
     .merge(FEAT, on=["icustay_id", "hr"], how="left"))
for g in FEATSETS:
    E[g] = GB_EVAL[g](E)
E = E[E.risk_main.notna()].reset_index(drop=True)
E["w"] = sample_weights(E, DP)
print(f"{len(E)} points, {E.y.sum()} positive; weighted prevalence {np.average(E.y, weights=E.w)*100:.2f}% "
      f"(population: {DP.y.mean()*100:.2f}%)")

g_e = E.icustay_id.values
E["agent_cal"] = crossfit_platt(logit(np.clip(E.risk_main, 0.005, 0.995)), E.y, g_e, E.w)
for s in SCORES:
    E[f"{s}_cal"] = crossfit_platt(E[s].values, E.y, g_e, E.w)

methods = {"Agent": "risk_main"} | {GB_NAMES[g]: g for g in FEATSETS} | {SCORE_NAMES[s]: s for s in SCORES}
sc = {k: E[v].values for k, v in methods.items()}
t_unw = boot_table(E.y, sc, g_e, ref="Agent")
t_w = boot_table(E.y, sc, g_e, w=E.w.values, ref="Agent")
print("UNWEIGHTED (enriched sample, comparable with Part 9):"); print(t_unw.to_string())
print("\nWEIGHTED to the population of eligible decision points:"); print(t_w.to_string())

cal_cols = {"Agent (Platt, cross-fitted)": "agent_cal", "Agent (raw verbal risk)": "risk_main"} | \
           {GB_NAMES[g]: g for g in FEATSETS} | {f"{SCORE_NAMES[s]} (Platt)": f"{s}_cal" for s in SCORES}
calib = pd.DataFrame({k: calib_stats(E.y, E[v], E.w) for k, v in cal_cols.items()}).T
print("\nCALIBRATION (weighted):"); print(calib.to_string())

rate_agent = float(np.average(E.alert_main, weights=E.w))
ops = {"Agent alert flag": op_point(E.y, E.alert_main == 1, E.w)}
for k, v in cal_cols.items():
    if k.startswith("Agent (raw"): continue
    ops[f"{k} @ agent alert rate"] = op_point(E.y, E[v] >= thr_for_rate(E[v], rate_agent, E.w), E.w)
for t in (0.05, 0.10):
    for k in ["Agent (Platt, cross-fitted)", GB_NAMES["gb_full"], GB_NAMES["gb_sofa"]]:
        ops[f"{k} @ risk ≥ {int(t*100)}%"] = op_point(E.y, E[cal_cols[k]] >= t, E.w)
ops["SOFA ≥ 2"] = op_point(E.y, E.sofa >= 2, E.w)
ops["SIRS ≥ 2"] = op_point(E.y, E.sirs >= 2, E.w)
ops["qSOFA ≥ 2"] = op_point(E.y, E.qsofa >= 2, E.w)
ops["NEWS2 ≥ 5"] = op_point(E.y, E.news2 >= 5, E.w)
ops = pd.DataFrame(ops).T
print(f"\nOPERATING POINTS (weighted; agent alert rate = {rate_agent:.3f}):"); print(ops.to_string())

t_unw.to_csv(f"{OUT_DIR}/ext_eval568_unweighted.csv"); t_w.to_csv(f"{OUT_DIR}/ext_eval568_weighted.csv")
calib.to_csv(f"{OUT_DIR}/ext_calibration.csv"); ops.to_csv(f"{OUT_DIR}/ext_operating_points.csv")
E.to_parquet(f"{OUT_DIR}/eval568_scored.parquet", index=False)

# ---- 12f. Decision-curve analysis and calibration plots ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
TH = np.round(np.arange(0.005, 0.2001, 0.005), 3)
prev = np.average(E.y, weights=E.w)
dca = pd.DataFrame({"threshold": TH, "Treat all": prev - (1 - prev) * TH / (1 - TH), "Treat none": 0.0})
for k in ["Agent (Platt, cross-fitted)", GB_NAMES["gb_sofa"], GB_NAMES["gb_matched"], GB_NAMES["gb_full"], "NEWS2 (Platt)", "SOFA total (Platt)"]:
    dca[k] = net_benefit(E.y, E[cal_cols[k]], E.w, TH)
dca.to_csv(f"{OUT_DIR}/ext_decision_curve.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for k in dca.columns[1:]:
    ax[0].plot(dca.threshold * 100, dca[k], label=k, lw=1.5 if k.startswith("Treat") else 2,
               ls="--" if k.startswith("Treat") else "-")
ax[0].set_ylim(-0.005, max(0.02, dca.iloc[:, 3:].max().max() * 1.2)); ax[0].set_xlabel("Threshold probability (%)")
ax[0].set_ylabel("Net benefit"); ax[0].legend(fontsize=8); ax[0].set_title("Decision curve (population-weighted)")
xmax = 0.05
for k in ["Agent (Platt, cross-fitted)", GB_NAMES["gb_full"], "Agent (raw verbal risk)"]:
    cb = pd.DataFrame({"p": E[cal_cols[k]].values, "y": E.y.values, "w": E.w.values})
    cb["bin"] = pd.qcut(cb.p.rank(method="first"), 5, labels=False)          # equal-count quintiles, robust to ties
    pts_ = cb.groupby("bin")[["p", "y", "w"]].apply(lambda d: pd.Series({"x": np.average(d.p, weights=d.w), "o": np.average(d.y, weights=d.w)}))
    ax[1].plot(pts_.x, pts_.o, "o-", label=k); xmax = max(xmax, float(pts_.x.max()), float(pts_.o.max()))
lim = min(1.0, xmax * 1.15)
ax[1].plot([0, 1], [0, 1], "k:", lw=1); ax[1].set_xlim(0, lim); ax[1].set_ylim(0, lim)
ax[1].set_xlabel("Predicted risk"); ax[1].set_ylabel("Observed (weighted)"); ax[1].legend(fontsize=8)
ax[1].set_title("Calibration, quintiles (population-weighted)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_dca_calibration.png", dpi=200); plt.close()
print(dca.iloc[[1, 3, 9, 19, 39]].round(4).to_string(index=False))
print("saved", f"{OUT_DIR}/fig_dca_calibration.png")

# ---- 12g. Does the agent add information to gradient boosting? ----
def stacked_oof(X, y, groups, n_rep=20, k=5):
    X = np.asarray(X, dtype=float); y = np.asarray(y).astype(int); groups = np.asarray(groups)
    ug = np.unique(groups); preds = np.zeros(len(y)); cnt = np.zeros(len(y))
    for r in range(n_rep):
        rng = np.random.default_rng(SEED + r); fold_of = dict(zip(ug, rng.permutation(len(ug)) % k))
        f = np.array([fold_of[g] for g in groups])
        for j in range(k):
            tr, te = f != j, f == j
            if te.sum() == 0 or len(np.unique(y[tr])) < 2: continue
            m = LogisticRegression(C=1e6, max_iter=2000).fit(X[tr], y[tr])
            preds[te] += m.predict_proba(X[te])[:, 1]; cnt[te] += 1
    return preds / np.maximum(cnt, 1)

def loglik(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9); return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

def nri_idi(y, p_old, p_new):
    y = np.asarray(y).astype(int); up, dn = p_new > p_old, p_new < p_old
    ev, ne = y == 1, y == 0
    nri = (up[ev].mean() - dn[ev].mean()) + (dn[ne].mean() - up[ne].mean())
    idi = (p_new[ev].mean() - p_old[ev].mean()) - (p_new[ne].mean() - p_old[ne].mean())
    return nri, idi

from scipy.stats import chi2
la = logit(np.clip(E.risk_main, 0.005, 0.995))
stack_rows, stack_cols = [], {}
for g in FEATSETS:
    lg = logit(E[g])
    p_gb = stacked_oof(lg.reshape(-1, 1), E.y, g_e)
    p_st = stacked_oof(np.column_stack([lg, la]), E.y, g_e)
    stack_cols[f"{GB_NAMES[g]} alone"] = p_gb; stack_cols[f"{GB_NAMES[g]} + agent"] = p_st
    # likelihood-ratio test on the full sample (naive: ignores clustering; see bootstrap CI of the coefficient)
    m0 = LogisticRegression(C=1e6, max_iter=2000).fit(lg.reshape(-1, 1), E.y)
    m1 = LogisticRegression(C=1e6, max_iter=2000).fit(np.column_stack([lg, la]), E.y)
    lr_stat = 2 * (loglik(E.y.values, m1.predict_proba(np.column_stack([lg, la]))[:, 1]) - loglik(E.y.values, m0.predict_proba(lg.reshape(-1, 1))[:, 1]))
    coefs = []
    for ix in boot_indices(g_e, 1000):
        yb = E.y.values[ix]
        if yb.min() == yb.max(): continue
        coefs.append(LogisticRegression(C=1e6, max_iter=2000).fit(np.column_stack([lg[ix], la[ix]]), yb).coef_[0, 1])
    nri, idi = nri_idi(E.y.values, p_gb, p_st)
    nri_b, idi_b = [], []
    for ix in boot_indices(g_e, 1000, seed=SEED + 7):
        if E.y.values[ix].min() == E.y.values[ix].max(): continue
        a_, b_ = nri_idi(E.y.values[ix], p_gb[ix], p_st[ix]); nri_b.append(a_); idi_b.append(b_)
    stack_rows.append({"base model": GB_NAMES[g],
                       "agent coefficient (95% CI)": f"{m1.coef_[0,1]:.2f} ({np.percentile(coefs, 2.5):.2f} to {np.percentile(coefs, 97.5):.2f})",
                       "LR chi2 (1 df)": round(lr_stat, 2), "LR p (naive)": round(float(chi2.sf(lr_stat, 1)), 4),
                       "NRI (95% CI)": f"{nri:.3f} ({np.percentile(nri_b, 2.5):.3f} to {np.percentile(nri_b, 97.5):.3f})",
                       "IDI (95% CI)": f"{idi:.4f} ({np.percentile(idi_b, 2.5):.4f} to {np.percentile(idi_b, 97.5):.4f})"})
stack_tab = pd.DataFrame(stack_rows).set_index("base model")
disc = pd.concat([boot_table(E.y, {k: v for k, v in stack_cols.items() if GB_NAMES[g] in k}, g_e, ref=f"{GB_NAMES[g]} alone")
                  for g in FEATSETS])
print("Discrimination of out-of-fold stacked predictions (unweighted):"); print(disc.to_string())
print("\nIncremental value of the agent:"); print(stack_tab.to_string())
disc.to_csv(f"{OUT_DIR}/ext_stacking_discrimination.csv"); stack_tab.to_csv(f"{OUT_DIR}/ext_stacking_increment.csv")

# ---- 12h. What sets the onset time? ----
od = con.execute("""
SELECT l.icustay_id, s.si_time, s.first_sofa_time, s.onset_time, si.abx_time, si.culture_time
FROM stay_labels l JOIN sepsis3 s USING (icustay_id) JOIN suspected_infection si ON si.hadm_id = l.hadm_id
WHERE l.stay_class = 'sepsis_in_icu'""").df()
od.columns = od.columns.str.lower()
for c in ["si_time", "first_sofa_time", "onset_time", "abx_time", "culture_time"]:
    od[c] = pd.to_datetime(od[c])
gap = (od.si_time - od.first_sofa_time).dt.total_seconds() / 3600
drivers = pd.Series({
    "ICU-onset septic stays": len(od),
    "onset set by SI time (SI after first SOFA>=2 hour), %": round(100 * (gap > 0).mean(), 1),
    "onset set by SOFA (SOFA>=2 first reached after SI), %": round(100 * (gap < 0).mean(), 1),
    "SI and SOFA in the same hour, %": round(100 * (gap == 0).mean(), 1),
    "median SI minus first-SOFA gap, h": round(gap.median(), 1),
    "SI time defined by the antibiotic (abx earlier than culture), %": round(100 * (od.abx_time <= od.culture_time).mean(), 1),
    "SI time at exactly midnight (date-only antibiotic), %": round(100 * ((od.si_time.dt.hour == 0) & (od.si_time.dt.minute == 0) & (od.si_time == od.abx_time)).mean(), 1)})
print(drivers.to_string())
drivers.to_csv(f"{OUT_DIR}/ext_onset_drivers.csv")


In [ ]:
# =====================================================================================
# PART 13. Label sensitivity analyses
# Labels re-derived under IV antibiotic times, acute SOFA rise, intubated GCS excluded,
# hospital-onset >= 48 h, and all combined; agent outputs re-scored.
# =====================================================================================
# ---- label variants: settings and alternative inputs (IV antibiotic times, GCS without intubated verbal) ----
N_BOOT_VAR = 1000
t0 = time.time()
GRID = con.execute("SELECT icustay_id, hadm_id, hr, hour_time, sofa, s_cns FROM sofa_hourly").df()
GRID.columns = GRID.columns.str.lower(); GRID["hour_time"] = pd.to_datetime(GRID.hour_time)
COH = con.execute("""SELECT c.icustay_id, c.hadm_id, c.intime, c.outtime, c.hospital_expire_flag, a.admittime
                     FROM cohort_final c JOIN admissions a ON a.hadm_id = c.hadm_id""").df()
COH.columns = COH.columns.str.lower()
for c in ["intime", "outtime", "admittime"]: COH[c] = pd.to_datetime(COH[c])

SI_MAIN = con.execute("SELECT hadm_id, suspected_infection_time AS si_time FROM suspected_infection").df()
SI_MAIN.columns = SI_MAIN.columns.str.lower(); SI_MAIN["si_time"] = pd.to_datetime(SI_MAIN.si_time)

# ---- suspected infection with timestamped IV antibiotic administrations where available ----
SI_IV = con.execute(f"""
WITH abx_mv AS (
  SELECT c.hadm_id, mv.starttime AS abx_time
  FROM inputevents_mv mv JOIN cohort_final c ON c.icustay_id = mv.icustay_id
  JOIN d_items d ON d.itemid = mv.itemid
  WHERE lower(d.category) LIKE 'antibiotic%' AND COALESCE(mv.statusdescription, '') <> 'Rewritten' AND mv.starttime IS NOT NULL
),
abx_rx AS (
  SELECT c.hadm_id, p.startdate AS abx_time
  FROM prescriptions p JOIN cohort_final c ON p.hadm_id = c.hadm_id
  WHERE regexp_matches(lower(p.drug), '{ABX}')
    AND (p.route IS NULL OR upper(p.route) NOT IN ('OU','OS','OD','AU','AS','AD','TP','IR'))
    AND p.startdate IS NOT NULL
    AND c.hadm_id NOT IN (SELECT hadm_id FROM abx_mv)
),
abx AS (SELECT * FROM abx_mv UNION ALL SELECT * FROM abx_rx),
cx AS (
  SELECT c.hadm_id, COALESCE(m.charttime, m.chartdate) AS culture_time
  FROM microbiologyevents m JOIN cohort_final c ON m.hadm_id = c.hadm_id
  WHERE COALESCE(m.charttime, m.chartdate) IS NOT NULL
),
pairs AS (
  SELECT a.hadm_id, LEAST(a.abx_time, x.culture_time) AS si_time
  FROM abx a JOIN cx x ON a.hadm_id = x.hadm_id
  WHERE (a.abx_time <= x.culture_time AND x.culture_time <= a.abx_time + INTERVAL 72 HOUR)
     OR (x.culture_time <= a.abx_time AND a.abx_time <= x.culture_time + INTERVAL 24 HOUR)
)
SELECT hadm_id, MIN(si_time) AS si_time, (SELECT COUNT(DISTINCT hadm_id) FROM abx_mv) AS n_mv_hadm
FROM pairs GROUP BY hadm_id""").df()
SI_IV.columns = SI_IV.columns.str.lower(); SI_IV["si_time"] = pd.to_datetime(SI_IV.si_time)
print(f"suspected infection: main {len(SI_MAIN):,} admissions | IV-antibiotic version {len(SI_IV):,} "
      f"({int(SI_IV.n_mv_hadm.iloc[0]):,} admissions have timestamped MetaVision antibiotics)")
SI_IV = SI_IV[["hadm_id", "si_time"]]

# ---- CNS score without intubated-verbal GCS rows ----
CNS_NOETT = con.execute(f"""
WITH g AS (
  SELECT ce.icustay_id, ce.charttime,
    MAX(CASE WHEN ce.itemid IN (184, 220739) THEN ce.valuenum END) AS eye,
    MAX(CASE WHEN ce.itemid IN (723, 223900) THEN ce.valuenum END) AS verbal,
    MAX(CASE WHEN ce.itemid IN (454, 223901) THEN ce.valuenum END) AS motor,
    MAX(CASE WHEN ce.itemid IN (723, 223900) AND (ce.value ILIKE '%ET/Trach%' OR ce.value ILIKE '%ETT%') THEN 1 ELSE 0 END) AS ett
  FROM chartevents ce
  WHERE ce.itemid IN (184, 220739, 723, 223900, 454, 223901) AND (ce.error IS NULL OR ce.error = 0)
    AND ce.icustay_id IN (SELECT icustay_id FROM cohort_final)
  GROUP BY 1, 2),
e AS (
  SELECT g.icustay_id, {HR('g.charttime')} AS hr, MIN(g.eye + g.verbal + g.motor) AS gcs
  FROM g JOIN cohort_final c ON c.icustay_id = g.icustay_id
  WHERE g.eye IS NOT NULL AND g.verbal IS NOT NULL AND g.motor IS NOT NULL AND g.ett = 0
    AND g.charttime >= c.intime AND g.charttime <= c.outtime
  GROUP BY 1, 2),
j AS (SELECT gr.icustay_id, gr.hr, e.gcs FROM hourly_grid gr LEFT JOIN e ON e.icustay_id = gr.icustay_id AND e.hr = gr.hr)
SELECT icustay_id, hr, MIN(gcs) OVER (PARTITION BY icustay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW) AS gcs_min_noett
FROM j""").df()
CNS_NOETT.columns = CNS_NOETT.columns.str.lower()
gm = CNS_NOETT.gcs_min_noett
CNS_NOETT["s_cns_noett"] = np.select([gm.isna(), gm < 6, gm < 10, gm < 13, gm < 15], [0, 4, 3, 2, 1], 0)
GRID = GRID.merge(CNS_NOETT[["icustay_id", "hr", "s_cns_noett"]], on=["icustay_id", "hr"], how="left")
GRID["sofa_noett"] = GRID.sofa - GRID.s_cns + GRID.s_cns_noett.fillna(0)
print(f"data loaded ({(time.time()-t0)/60:.1f} min) | mean CNS points: as charted {GRID.s_cns.mean():.2f}, "
      f"intubated rows excluded {GRID.s_cns_noett.mean():.2f}")

# ---- label variants: derive labels and re-score all methods ----
def derive_labels(si, sofa_col="sofa", acute=False, hosp48=False):
    """Sepsis-3 onset per stay, same logic as Part 4 (window -48 h to +24 h around SI; onset = later of the two)."""
    g = GRID[["icustay_id", "hadm_id", "hr", "hour_time", sofa_col]].rename(columns={sofa_col: "s"})
    if acute:
        g = g.sort_values(["icustay_id", "hr"]).reset_index(drop=True)
        base = g.groupby("icustay_id")["s"].rolling(73, min_periods=1).min().reset_index(level=0, drop=True)
        g["qual"] = (g.s - base.sort_index()) >= 2
    else:
        g["qual"] = g.s >= 2
    q = g.loc[g.qual, ["icustay_id", "hadm_id", "hour_time"]].merge(si, on="hadm_id")
    q = q[(q.hour_time >= q.si_time - pd.Timedelta(hours=48)) & (q.hour_time <= q.si_time + pd.Timedelta(hours=24))]
    f = q.groupby("icustay_id").agg(first_sofa_time=("hour_time", "min"), si_time=("si_time", "first")).reset_index()
    f = f.merge(COH[["icustay_id", "intime", "outtime"]], on="icustay_id")
    f["onset_time"] = f[["si_time", "first_sofa_time"]].max(axis=1)
    f = f[f.onset_time <= f.outtime]
    f["onset_hr"] = np.ceil((f.onset_time - f.intime).dt.total_seconds() / 3600).astype(int)
    lab = COH.merge(f[["icustay_id", "onset_time", "onset_hr", "si_time", "first_sofa_time"]], on="icustay_id", how="left")
    early = lab.onset_hr < MIN_ONSET_HR
    if hosp48:
        early = early | (lab.onset_time < lab.admittime + pd.Timedelta(hours=48))
    lab["stay_class"] = np.where(lab.onset_hr.isna(), "no_sepsis", np.where(early, "present_on_admission", "sepsis_in_icu"))
    return lab

def decision_points_for(lab):
    d = GRID.loc[(GRID.hr >= MIN_ONSET_HR) & (GRID.hr % DECISION_EVERY_H == 0), ["icustay_id", "hr", "sofa", "sofa_noett"]]
    d = d.merge(lab[["icustay_id", "onset_hr", "stay_class"]], on="icustay_id")
    d = d[(d.stay_class != "present_on_admission") & (d.onset_hr.isna() | (d.hr < d.onset_hr))].copy()
    d["y"] = ((d.onset_hr - d.hr >= 1) & (d.onset_hr - d.hr <= HORIZON_H)).astype(int)
    return d

VARIANTS = {
    "main":       dict(si=SI_MAIN, sofa_col="sofa"),
    "iv_abx":     dict(si=SI_IV,   sofa_col="sofa"),
    "acute_sofa": dict(si=SI_MAIN, sofa_col="sofa", acute=True),
    "gcs_noett":  dict(si=SI_MAIN, sofa_col="sofa_noett"),
    "hosp48":     dict(si=SI_MAIN, sofa_col="sofa", hosp48=True),
    "strict":     dict(si=SI_IV,   sofa_col="sofa_noett", acute=True, hosp48=True),
}
VAR_NAMES = {"main": "Main (reproduced)", "iv_abx": "IV antibiotic times", "acute_sofa": "Acute SOFA rise ≥2 (72 h baseline)",
             "gcs_noett": "Intubated GCS excluded", "hosp48": "Hospital-onset ≥48 h", "strict": "All four combined"}

eval_keys = E[["icustay_id", "hr", "risk_main"]]       # E from Part 12e (agent answers on the 568 points)
var_rows, var_auc = [], []
for name, cfg in VARIANTS.items():
    t0 = time.time()
    lab = derive_labels(**cfg)
    cls = lab.groupby("stay_class").agg(n=("icustay_id", "size"), mortality=("hospital_expire_flag", "mean"),
                                         median_onset_hr=("onset_hr", "median"))
    sep = lab[lab.stay_class == "sepsis_in_icu"]
    gap = (sep.si_time - sep.first_sofa_time).dt.total_seconds()
    d = decision_points_for(lab)
    row = {"variant": VAR_NAMES[name],
           "no sepsis": int(cls.n.get("no_sepsis", 0)), "early/POA (excluded)": int(cls.n.get("present_on_admission", 0)),
           "ICU-onset (target)": int(cls.n.get("sepsis_in_icu", 0)),
           "ICU-onset mortality %": round(100 * cls.mortality.get("sepsis_in_icu", np.nan), 1),
           "median onset h": cls.median_onset_hr.get("sepsis_in_icu", np.nan),
           "onset set by SI time %": round(100 * (gap > 0).mean(), 1) if len(sep) else np.nan,
           "decision points": len(d), "positive %": round(100 * d.y.mean(), 2)}
    # ---- re-score the 568 points ----
    ev = eval_keys.merge(d, on=["icustay_id", "hr"], how="inner")
    tr = d[~d.icustay_id.isin(EVAL_STAYS)].merge(FEAT[["icustay_id", "hr"] + [f for f in FEATS if f != "hr"]], on=["icustay_id", "hr"])
    gbv = make_gb(GB_PARAMS["gb_sofa"]).fit(tr.sort_values(["icustay_id", "hr"])[FEATS], tr.sort_values(["icustay_id", "hr"]).y)
    evf = ev.merge(FEAT[["icustay_id", "hr"] + [f for f in FEATS if f != "hr"]], on=["icustay_id", "hr"])
    ev = ev.assign(gb=gbv.predict_proba(evf[FEATS])[:, 1])
    sofa_v = ev["sofa_noett"] if cfg["sofa_col"] == "sofa_noett" else ev["sofa"]
    row.update({"eligible eval points": len(ev), "eval positives": int(ev.y.sum())})
    var_rows.append(row)
    if ev.y.nunique() == 2 and ev.y.sum() >= 10:
        bt = boot_table(ev.y, {"Agent": ev.risk_main.values, "SOFA": sofa_v.values, "GB (SOFA inputs, retrained)": ev.gb.values},
                        ev.icustay_id.values, ref="Agent", n_boot=N_BOOT_VAR, metrics=("AUROC",))
        bt.insert(0, "variant", VAR_NAMES[name]); var_auc.append(bt)
    print(f"  {name} done ({(time.time()-t0)/60:.1f} min)", flush=True)

var_tab = pd.DataFrame(var_rows).set_index("variant")
var_auc = pd.concat(var_auc)
print("Cohort under each label definition:"); print(var_tab.to_string())
print("\nRe-scored evaluation points (AUROC, stay-bootstrap 95% CI):"); print(var_auc.to_string())
chk = con.execute("SELECT stay_class, COUNT(*) FROM stay_labels GROUP BY 1").df()
print("\nSanity check, Part 4 counts:", dict(zip(chk.iloc[:, 0], chk.iloc[:, 1])))
var_tab.to_csv(f"{OUT_DIR}/ext_label_variants_cohort.csv"); var_auc.to_csv(f"{OUT_DIR}/ext_label_variants_auroc.csv")


In [ ]:
# =====================================================================================
# PART 14A. New agent runs: repeated seeds, trends agent, hybrid agent
# Generalised runner, extra tools (trends, calibrated GB risk) with leakage test,
# runs on the 568 points and their evaluation.
# =====================================================================================
# ---- Part 14 setup ----
from openai import OpenAI
client = globals().get("client") or OpenAI(base_url=VLLM_URL, api_key="EMPTY", timeout=600)
if "sampling_for" not in globals():
    def sampling_for(thinking):
        return (dict(temperature=0.6, top_p=0.95, extra={"top_k": 20}) if thinking
                else dict(temperature=0.7, top_p=0.8, extra={"top_k": 20}))
ALL_TOOLS = globals().get("ALL_TOOLS") or {t["function"]["name"]: t for t in TOOLS}
BASE_TOOLS = ["get_vitals", "get_labs", "get_urine_output", "search_notes"]
def vllm_ready(timeout=10):
    """Quick health check: is the vLLM server up, serving MODEL, and able to answer a 5-token request within 60 s?"""
    import urllib.request
    try:
        with urllib.request.urlopen(VLLM_URL.rstrip("/") + "/models", timeout=timeout) as r:
            models = [m["id"] for m in json.load(r)["data"]]
    except Exception as e:
        print(f"vLLM is not reachable at {VLLM_URL} ({type(e).__name__}: {e}). Start the server, then re-run this cell.")
        return False
    if MODEL not in models:
        print(f"vLLM is up but serves {models}, not {MODEL!r}. Fix MODEL in Part 0 or restart vLLM with that model.")
        return False
    t0 = time.time()
    try:
        client.with_options(timeout=60, max_retries=0).chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "Reply with OK."}], max_tokens=5,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    except Exception as e:
        print(f"vLLM is up but a 5-token request did not finish within 60 s ({type(e).__name__}). "
              "It is probably busy with another job (requests queue behind it). Run the LLM parts when it is free.")
        return False
    print(f"vLLM ready: {MODEL} answered a 5-token request in {time.time() - t0:.1f} s")
    return True

def run_agent_x(env, tools, system=None, thinking=True, max_steps=8, seed_base=SEED, client_=None, model_=None, sampling=None):
    """run_agent (Part 7) with the tool list, system prompt, seed, client and model as arguments.
    thinking=None omits the Qwen-specific chat_template_kwargs (for other model families)."""
    client_ = client_ or client; model_ = model_ or MODEL; system = system or SYSTEM
    samp = sampling or sampling_for(bool(thinking))
    allowed = [t["function"]["name"] for t in tools]
    extra = dict(samp.get("extra", {}))
    if thinking is not None: extra["chat_template_kwargs"] = {"enable_thinking": bool(thinking)}
    user = patient_header(env) if "patient_header" in globals() else (
        f"Patient: {int(env.age)}-year-old {'man' if env.gender == 'M' else 'woman'}, admitted to {env.unit}. "
        f"ICU admission time: {_t(env.intime)}. Current time: {_t(env.now)} "
        f"({(env.now - pd.Timestamp(env.intime)).total_seconds() / 3600:.0f} hours into the ICU stay). "
        "Assess the risk of sepsis onset in the next 12 hours.")
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    trace, n_tools, ptok, ctok, final, t0 = [], 0, 0, 0, None, time.time()
    retried, force_final, n_trunc = False, False, 0
    for step in range(max_steps + 2):
        if step >= max_steps and not force_final:
            force_final = True
            messages.append({"role": "user", "content": "Give your final JSON answer now."})
        use_tools = None if (force_final or not tools) else tools
        resp = client_.chat.completions.create(
            model=model_, messages=messages, max_tokens=MAX_TOKENS, temperature=samp["temperature"], top_p=samp["top_p"],
            seed=seed_base + step, tools=use_tools, tool_choice="auto" if use_tools else None, extra_body=extra)
        choice = resp.choices[0]; msg = choice.message
        n_trunc += choice.finish_reason == "length"
        ptok += resp.usage.prompt_tokens; ctok += resp.usage.completion_tokens
        reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
        if not msg.tool_calls:
            final = parse_final(msg.content)
            trace.append({"step": step, "type": "final", "content": msg.content, "reasoning": reasoning, "finish_reason": choice.finish_reason})
            if final is None and not retried:
                retried, force_final = True, True
                messages.append({"role": "assistant", "content": msg.content or ""})
                messages.append({"role": "user", "content": "Return ONLY the final JSON object in the required format, nothing else."})
                continue
            break
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                        for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            n_tools += 1
            try:
                args = json.loads(tc.function.arguments or "{}")
                result = getattr(env, tc.function.name)(**args) if tc.function.name in allowed else {"error": f"unknown tool {tc.function.name}"}
            except Exception as e:
                args, result = tc.function.arguments, {"error": f"{type(e).__name__}: {e}"}
            out = json.dumps(result, default=str); env.tool_log.append(out)
            trace.append({"step": step, "type": "tool", "name": tc.function.name, "args": args, "result": out,
                          "reasoning": reasoning, "finish_reason": choice.finish_reason})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return {"final": final, "trace": trace, "n_tool_calls": n_tools, "prompt_tokens": ptok, "completion_tokens": ctok,
            "json_retry": retried, "n_truncated": int(n_trunc), "latency_s": round(time.time() - t0, 1), "tool_log": env.tool_log}

def run_config(name, points, cfg, path=None):
    """Run one configuration on a set of decision points; appends to JSONL; resumable."""
    path = path or f"{OUT_DIR}/x_{name}.jsonl"
    done = {(int(d["icustay_id"]), int(d["hr"])) for d in read_results(path) if d.get("final") is not None}   # failed records are retried
    todo = [r for r in points.itertuples() if (int(r.icustay_id), int(r.hr)) not in done]
    print(f"=== {name}: {len(done)} done, {len(todo)} to run -> {path}", flush=True)
    if not todo: return path
    lock = threading.Lock(); env_cls = cfg.get("env_cls", Env)
    def work(r):
        try:
            env = env_cls(con.cursor(), r.icustay_id, r.hour_time)
            if cfg.get("mode", "agent") == "single":
                out = run_single(env, cfg.get("thinking", True))
            else:
                out = run_agent_x(env, [ALL_TOOLS[n] for n in cfg["tools"]], cfg.get("system"), cfg.get("thinking", True),
                                  seed_base=cfg.get("seed_base", SEED), client_=cfg.get("client"), model_=cfg.get("model"),
                                  sampling=cfg.get("sampling"))
        except Exception as e:
            out = {"final": None, "trace": [{"type": "error", "content": f"{type(e).__name__}: {e}"}], "n_tool_calls": 0,
                   "prompt_tokens": 0, "completion_tokens": 0, "latency_s": 0, "tool_log": [], "json_retry": False, "n_truncated": 0}
        rec = {"icustay_id": int(r.icustay_id), "hr": int(r.hr), "y": int(r.y), "run": name, **out}
        with lock:
            with open(path, "a") as f: f.write(json.dumps(rec, default=str) + "\n")
        return rec
    t0, ok = time.time(), 0
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        futs = [ex.submit(work, r) for r in todo]
        for i, f in enumerate(as_completed(futs), 1):
            ok += f.result()["final"] is not None
            if i % 25 == 0 or i == len(futs):
                print(f"  {i}/{len(futs)} | valid {ok} | {(time.time() - t0) / 60:.1f} min", flush=True)
    return path

# ---- 14a, new tools ----
TREND_LABS = ["lactate", "wbc", "creatinine", "platelets", "bilirubin", "bicarbonate"]
GB_RISK, GB_RISK_REF = {}, np.sort(ALLP["gb_full"].values)
GB_PREV = float(DP.y.mean())

class EnvPlus(Env):
    def get_trends(self, hours_back=24):
        hours_back = min(max(float(hours_back), 8), 72)
        df = _df(self.cur.execute("""
            SELECT itemid, charttime, valuenum FROM chartevents
            WHERE icustay_id = ? AND itemid IN (SELECT UNNEST(?)) AND valuenum IS NOT NULL AND (error IS NULL OR error = 0)
              AND charttime <= ? AND charttime > ? AND (storetime IS NULL OR storetime <= ?)""",
            [self.icustay_id, list(ITEM2VITAL), self.now, self._since(hours_back), self.now]))
        out = {"window_hours": hours_back, "vitals": {}, "labs": {}}
        if len(df):
            self._mark(df.charttime)
            df["vital"] = df.itemid.map(ITEM2VITAL)
            df.loc[df.vital == "temp_f", "valuenum"] = (df.valuenum - 32) * 5 / 9
            df["vital"] = df.vital.replace({"temp_f": "temp_c"})
            df = df[(df.valuenum > 0) & (df.valuenum < 400)]
            recent = df.charttime > self._since(6)
            for v, g in df.groupby("vital"):
                r, e = g[recent.loc[g.index]], g[~recent.loc[g.index]]
                out["vitals"][v] = {"mean_last_6h": round(float(r.valuenum.mean()), 1) if len(r) else None,
                                    "mean_earlier": round(float(e.valuenum.mean()), 1) if len(e) else None,
                                    "change": round(float(r.valuenum.mean() - e.valuenum.mean()), 1) if len(r) and len(e) else None}
        for name in TREND_LABS:
            l = _df(self.cur.execute("""SELECT charttime, valuenum FROM labevents
                    WHERE hadm_id = ? AND itemid = ? AND valuenum IS NOT NULL AND charttime <= ? AND charttime > ?
                    ORDER BY charttime DESC LIMIT 2""", [self.hadm_id, LABS[name], self.now, self._since(max(hours_back, 48))]))
            if l.empty: out["labs"][name] = "no result"; continue
            self._mark(l.charttime)
            item = {"latest": round(float(l.valuenum.iloc[0]), 2), "latest_time": _t(l.charttime.iloc[0])}
            if len(l) == 2:
                item.update({"previous": round(float(l.valuenum.iloc[1]), 2),
                             "change": round(float(l.valuenum.iloc[0] - l.valuenum.iloc[1]), 2),
                             "hours_between": round((pd.Timestamp(l.charttime.iloc[0]) - pd.Timestamp(l.charttime.iloc[1])).total_seconds() / 3600, 1)})
            out["labs"][name] = item
        return out

    def get_gb_risk(self):
        hr = int(round((self.now - pd.Timestamp(self.intime)).total_seconds() / 3600))
        p = GB_RISK.get((self.icustay_id, hr))
        if p is None: return {"note": "model risk not available at this time"}
        self.seen.append(self.now)
        return {"model": "gradient boosting on vital signs, labs, urine output and SOFA inputs (no antibiotics or cultures), calibrated",
                "risk_next_12h": round(float(p), 3),
                "percentile_among_icu_decision_points": round(100 * float(np.searchsorted(GB_RISK_REF, p) / len(GB_RISK_REF)), 1),
                "average_risk_next_12h": round(GB_PREV, 3)}

ALL_TOOLS["get_trends"] = _fn("get_trends", "Recent changes: vital-sign means in the last 6 h vs earlier in the window, and the last two values of lactate, WBC, creatinine, platelets, bilirubin, bicarbonate.",
                              {"hours_back": {"type": "number", "description": "window in hours (default 24, max 72)"}})
ALL_TOOLS["get_gb_risk"] = _fn("get_gb_risk", "Calibrated risk of sepsis onset in the next 12 h from a validated gradient-boosting model, with its percentile among ICU patients.", {})
ALL_TOOLS.setdefault("get_sofa", _fn("get_sofa", "Current SOFA score and its six components.", {}))
ALL_TOOLS.setdefault("get_infection_data", _fn("get_infection_data", "Antibiotics and cultures ordered so far.", {}))

SYSTEM_TRENDS = SYSTEM + "\n- get_trends summarises how vital signs and key labs have changed recently; use it to judge direction, not only level."
SYSTEM_HYBRID = SYSTEM_TRENDS + (
    f"\n- get_gb_risk returns the calibrated risk from a validated statistical model. The average risk of sepsis onset in the next 12 hours"
    f" for an ICU patient is about {100 * GB_PREV:.1f}%. Use the model's risk as your starting point and move it up or down only when the"
    " clinical evidence (trends, labs, notes) clearly justifies it. In the evidence, state the model risk and why you adjusted it."
    " Your sepsis_risk must be a calibrated probability.")

# GB risks for the 568 points: gb_full trained on the other stays (Part 12d)
GB_RISK.update({(int(a), int(b)): float(c) for a, b, c in zip(E.icustay_id, E.hr, E.gb_full)})

def leakage_test(points, env_cls=EnvPlus, n=30, seed=3):
    cur = con.cursor(); bad = calls = 0
    for r in points.sample(min(n, len(points)), random_state=seed).itertuples():
        env = env_cls(cur, r.icustay_id, r.hour_time); env.get_trends(48); env.get_gb_risk(); calls += 1
        bad += sum(t > env.now for t in env.seen)
    assert bad == 0, "TIME-GATING VIOLATION in get_trends / get_gb_risk"
    print(f"PASS: {calls} points, get_trends and get_gb_risk return no future timestamps")
leakage_test(eval_points)
print(json.dumps(EnvPlus(con.cursor(), eval_points.iloc[0].icustay_id, eval_points.iloc[0].hour_time).get_trends(24), default=str)[:700])

# ---- 14b–14c. Repeated runs, trends agent, hybrid agent (568 points) ----
X_CONFIGS = {
    "rep1":   dict(tools=BASE_TOOLS, thinking=True, seed_base=SEED + 1000),
    "rep2":   dict(tools=BASE_TOOLS, thinking=True, seed_base=SEED + 2000),
    "rep3":   dict(tools=BASE_TOOLS, thinking=True, seed_base=SEED + 3000),
    "trends": dict(tools=BASE_TOOLS + ["get_trends"], thinking=True, env_cls=EnvPlus, system=SYSTEM_TRENDS),
    "hybrid": dict(tools=BASE_TOOLS + ["get_trends", "get_gb_risk"], thinking=True, env_cls=EnvPlus, system=SYSTEM_HYBRID),
}
RUN_X = ["rep1", "rep2", "rep3", "trends", "hybrid"]      # remove any you do not want
X_LIMIT = None                                            # e.g. 24 for a quick trial
pts568 = eval_points if not X_LIMIT else eval_points.sample(X_LIMIT, random_state=SEED)
assert vllm_ready(), "vLLM is not ready (see the message above); nothing was run"
for name in RUN_X:
    run_config(name, pts568, X_CONFIGS[name])
print("all selected configurations finished")

# ---- 14d. Evaluation of the new runs ----
def faithful_share(rows):
    num_re = re.compile(r"(?<![\w.])-?\d+(?:\.\d+)?")
    ts_re = re.compile(r"\d{4}-\d{2}-\d{2}(?:[ T]\d{1,2}:\d{2}(?::\d{2})?)?|\b\d{1,2}:\d{2}(?::\d{2})?\b")
    nums = lambda s: [float(x) for x in num_re.findall(ts_re.sub(" ", s))]
    chk = sup = 0
    for r in rows:
        if not r["final"]: continue
        tn = np.array(sorted(set(n for t in r["tool_log"] for n in nums(t))) or [np.nan])
        for e in r["final"].get("evidence", []):
            for n in nums(str(e)):
                if abs(n) < 1 or n in (12.0, 24.0, 48.0): continue
                chk += 1; sup += bool(np.any(np.isclose(tn, n, rtol=0.01, atol=0.051)))
    return sup / max(chk, 1)

keys568 = set(zip(eval_points.icustay_id.astype(int), eval_points.hr.astype(int)))
X = E[["icustay_id", "hr", "y", "w", "risk_main", "alert_main", "gb_full", "gb_sofa", "sofa", "news2"]].copy()
xrows = {"main": main_rows}
for name in X_CONFIGS:
    p = f"{OUT_DIR}/x_{name}.jsonl"
    if os.path.exists(p):
        xrows[name] = read_results(p, keys568)
        X = X.merge(results_frame(xrows[name], name), on=["icustay_id", "hr"], how="left")
print("runs found:", list(xrows))

# ---- repeatability ----
reps = [c for c in ["risk_main", "risk_rep1", "risk_rep2", "risk_rep3"] if c in X]
if len(reps) > 1:
    R = X[reps].dropna()
    k = len(reps); n = len(R); grand = R.values.mean()
    msb = k * ((R.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    msw = ((R.values - R.mean(axis=1).values[:, None]) ** 2).sum() / (n * (k - 1))
    icc1 = (msb - msw) / (msb + (k - 1) * msw)
    A = X[[c.replace("risk", "alert") for c in reps]].dropna().astype(int).values
    pj = A.mean(axis=0).mean(); P_i = ((A.sum(1) ** 2 - A.sum(1)) / (k * (k - 1)) + (((k - A.sum(1)) ** 2 - (k - A.sum(1))) / (k * (k - 1))))
    pe = pj ** 2 + (1 - pj) ** 2; fleiss = (P_i.mean() - pe) / (1 - pe)
    X["risk_mean4"] = X[reps].mean(axis=1)
    rep_tab = boot_table(X.y, {c: X[c].values for c in reps + ["risk_mean4"]}, X.icustay_id.values, ref="risk_main", metrics=("AUROC", "AUPRC"))
    print(f"Repeatability over {k} runs, {n} points: mean per-point SD of risk {R.std(axis=1).mean():.3f}; "
          f"ICC(1) {icc1:.2f}; Fleiss' kappa for the alert flag {fleiss:.2f}")
    print(rep_tab.to_string()); rep_tab.to_csv(f"{OUT_DIR}/ext_repeatability.csv")

# ---- trends and hybrid ----
cmp_cols = {"Agent (main)": "risk_main", "GB, full": "gb_full"}
for name in ["trends", "hybrid"]:
    if f"risk_{name}" in X: cmp_cols[f"Agent ({name})"] = f"risk_{name}"
if "risk_mean4" in X: cmp_cols["Agent (mean of 4 runs)"] = "risk_mean4"
Xc = X.dropna(subset=list(cmp_cols.values()))
for ref in ["Agent (main)", "GB, full"]:
    t = boot_table(Xc.y, {k: Xc[v].values for k, v in cmp_cols.items()}, Xc.icustay_id.values, w=Xc.w.values, ref=ref)
    print(f"\nWEIGHTED, paired vs {ref} ({len(Xc)} points):"); print(t.to_string())
    t.to_csv(f"{OUT_DIR}/ext_newruns_vs_{'agent' if ref.startswith('Agent') else 'gb'}.csv")
summ = []
for name, rows_n in xrows.items():
    if name not in ("main", "trends", "hybrid") and not name.startswith("rep"): continue
    rr = pd.DataFrame(rows_n); col = f"risk_{name}"; m = X[col].notna()
    cal = X.loc[m, col]
    summ.append({"run": name, "valid %": round(100 * m.mean(), 1),
                 **{f"{k} (weighted)": v for k, v in calib_stats(X.loc[m, "y"], np.clip(cal, 1e-4, 1 - 1e-4), X.loc[m, "w"]).items() if k in ("Brier", "O/E", "slope")},
                 **op_point(X.loc[m, "y"], X.loc[m, col.replace("risk", "alert")] == 1, X.loc[m, "w"]),
                 "tool calls": round(rr.n_tool_calls.mean(), 1), "tokens": int((rr.prompt_tokens + rr.completion_tokens).mean()),
                 "latency s": round(rr.latency_s.mean(), 1), "faithful %": round(100 * faithful_share(rows_n), 1)})
summ = pd.DataFrame(summ).set_index("run")
print("\nRaw verbal risk calibration and own-alert operating point (weighted):"); print(summ.to_string())
summ.to_csv(f"{OUT_DIR}/ext_newruns_summary.csv")
if "hybrid" in xrows:
    hy = X.dropna(subset=["risk_hybrid"])
    moved = (hy.risk_hybrid - hy.gb_full)
    print(f"\nHybrid: mean adjustment of the model risk {moved.mean():+.3f}; adjusted up in {100*(moved > 0.005).mean():.0f}% "
          f"and down in {100*(moved < -0.005).mean():.0f}% of points; Spearman r with GB "
          f"{hy[['risk_hybrid', 'gb_full']].corr(method='spearman').iloc[0, 1]:.2f}")


In [ ]:
# =====================================================================================
# PART 14B. Independent replication sample and full trajectories
# 500 new stays (up to 2 points each) and 48-h trajectories in 120 stays; GB re-fitted
# without these stays; evaluation and lead time at matched alert burden.
# =====================================================================================
# sizes: ~1,000 replication + ~1,400 trajectory cases per configuration (~9 h each at ~4.5 cases/min)
# NOTE: the samples are frozen in eval_points_large.csv / eval_points_traj.csv the first time this cell runs;
#       to change the sizes later, delete those two files (and any x_*_large / x_*_traj .jsonl files) first.
LARGE_PER_CLASS, LARGE_POINTS_PER_STAY = 250, 2
TRAJ_SEPTIC, TRAJ_NONSEPTIC, TRAJ_WINDOW_H = 60, 60, 48
LARGE_CONFIGS = ["main", "hybrid"]   # runs main first (large, then traj), then hybrid; interrupt after main if short of time
RUN_LARGE, RUN_TRAJ = True, True
CFG_ALL = {"main": dict(tools=BASE_TOOLS, thinking=True)} | X_CONFIGS

rng_l = np.random.default_rng(SEED + 11)
dp_free = DP[~DP.icustay_id.isin(EVAL_STAYS)]
st_free = dp_free.groupby("icustay_id").stay_class.first()
L_CSV, T_CSV = f"{OUT_DIR}/eval_points_large.csv", f"{OUT_DIR}/eval_points_traj.csv"
pts_all = con.execute("SELECT * FROM decision_points").df(); pts_all.columns = pts_all.columns.str.lower()
if not os.path.exists(L_CSV):
    pool_p, pool_n = st_free[st_free == "sepsis_in_icu"].index.values, st_free[st_free == "no_sepsis"].index.values
    pos = rng_l.choice(pool_p, min(LARGE_PER_CLASS, len(pool_p) // 2), replace=False)   # leave stays for the trajectories
    neg = rng_l.choice(pool_n, min(LARGE_PER_CLASS, len(pool_n) // 2), replace=False)
    big = pts_all[pts_all.icustay_id.isin(set(pos) | set(neg))]
    big = big.sample(frac=1, random_state=SEED).groupby("icustay_id").head(LARGE_POINTS_PER_STAY).sort_values(["icustay_id", "hr"])
    big.to_csv(L_CSV, index=False)
LARGE = pd.read_csv(L_CSV, parse_dates=["hour_time"])
if not os.path.exists(T_CSV):
    used = EVAL_STAYS | set(LARGE.icustay_id)
    free = st_free[~st_free.index.isin(used)]
    pool_p, pool_n = free[free == "sepsis_in_icu"].index.values, free[free == "no_sepsis"].index.values
    sp = rng_l.choice(pool_p, min(TRAJ_SEPTIC, len(pool_p) // 2), replace=False)
    sn = rng_l.choice(pool_n, min(TRAJ_NONSEPTIC, len(pool_n) // 2), replace=False)
    a = pts_all[pts_all.icustay_id.isin(set(sp))]
    a = a[a.hr >= a.onset_hr - TRAJ_WINDOW_H]
    b = pts_all[pts_all.icustay_id.isin(set(sn))].copy()
    start = b.groupby("icustay_id").hr.agg(lambda h: rng_l.choice(h.values[h.values <= max(h.min(), h.max() - TRAJ_WINDOW_H)]))
    b = b[(b.hr >= b.icustay_id.map(start)) & (b.hr < b.icustay_id.map(start) + TRAJ_WINDOW_H)]
    pd.concat([a, b]).sort_values(["icustay_id", "hr"]).to_csv(T_CSV, index=False)
TRAJ = pd.read_csv(T_CSV, parse_dates=["hour_time"])
print(f"replication sample: {len(LARGE)} points from {LARGE.icustay_id.nunique()} stays ({100*LARGE.y.mean():.1f}% positive) | "
      f"trajectories: {len(TRAJ)} points from {TRAJ.icustay_id.nunique()} stays")
eta = lambda n: n * 95.1 / CONCURRENCY / 3600
print(f"estimated time per configuration: replication {eta(len(LARGE)):.1f} h, trajectories {eta(len(TRAJ)):.1f} h")

# GB (gb_full) re-fitted without the new stays, used both as comparator and as the hybrid agent's tool
NEW_STAYS = set(LARGE.icustay_id) | set(TRAJ.icustay_id)
TRAIN_L = DPF[~DPF.icustay_id.isin(EVAL_STAYS | NEW_STAYS)]
print(f"GB training set for 14e: {TRAIN_L.icustay_id.nunique():,} stays")
assert TRAIN_L.icustay_id.nunique() >= 50, "too few training stays left; reduce LARGE_PER_CLASS / TRAJ_*"
GB_L = {g: fit_calibrated(TRAIN_L, FEATSETS[g], GB_PARAMS[g]) for g in ("gb_sofa", "gb_full")}
for P in (LARGE, TRAJ):
    F_ = P[["icustay_id", "hr"]].merge(FEAT, on=["icustay_id", "hr"], how="left")
    for g in GB_L: P[g] = GB_L[g](F_)
    GB_RISK.update({(int(a), int(b)): float(c) for a, b, c in zip(P.icustay_id, P.hr, P.gb_full)})
leakage_test(LARGE)

assert vllm_ready(), "vLLM is not ready (see the message above); nothing was run"
for cfg in LARGE_CONFIGS:
    if RUN_LARGE: run_config(f"{cfg}_large", LARGE, CFG_ALL[cfg])
    if RUN_TRAJ:  run_config(f"{cfg}_traj", TRAJ, CFG_ALL[cfg])

# ---- evaluate the replication sample ----
kL = set(zip(LARGE.icustay_id.astype(int), LARGE.hr.astype(int)))
EL = LARGE[["icustay_id", "hr", "y", "stay_class", "onset_hr", "gb_sofa", "gb_full"]].merge(FEAT[["icustay_id", "hr"] + SCORES], on=["icustay_id", "hr"])
for cfg in LARGE_CONFIGS:
    rows_c = read_results(f"{OUT_DIR}/x_{cfg}_large.jsonl", kL)
    if rows_c: EL = EL.merge(results_frame(rows_c, cfg), on=["icustay_id", "hr"], how="left")
EL["w"] = sample_weights(EL, DP)
cols = {f"Agent ({c})": f"risk_{c}" for c in LARGE_CONFIGS if f"risk_{c}" in EL} | \
       {GB_NAMES["gb_sofa"]: "gb_sofa", GB_NAMES["gb_full"]: "gb_full"} | {SCORE_NAMES[s]: s for s in SCORES}
ELc = EL.dropna(subset=list(cols.values()))
if len(ELc):
    ref = next(iter(cols))
    tL = boot_table(ELc.y, {k: ELc[v].values for k, v in cols.items()}, ELc.icustay_id.values, ref=ref)
    tLw = boot_table(ELc.y, {k: ELc[v].values for k, v in cols.items()}, ELc.icustay_id.values, w=ELc.w.values, ref=ref)
    print(f"REPLICATION SAMPLE ({len(ELc)} points, {ELc.y.sum()} positive) - unweighted:"); print(tL.to_string())
    print("weighted to the population:"); print(tLw.to_string())
    tL.to_csv(f"{OUT_DIR}/ext_replication_unweighted.csv"); tLw.to_csv(f"{OUT_DIR}/ext_replication_weighted.csv")

# ---- trajectories: lead time and alert burden ----
kT = set(zip(TRAJ.icustay_id.astype(int), TRAJ.hr.astype(int)))
ET = TRAJ[["icustay_id", "hr", "y", "stay_class", "onset_hr", "gb_full"]].copy()
traj_out = []
for cfg in LARGE_CONFIGS:
    rows_c = read_results(f"{OUT_DIR}/x_{cfg}_traj.jsonl", kT)
    if not rows_c: continue
    T = ET.merge(results_frame(rows_c, cfg), on=["icustay_id", "hr"], how="left").dropna(subset=[f"alert_{cfg}"])
    neg = T[T.stay_class == "no_sepsis"]; pos = T[T.stay_class == "sepsis_in_icu"]
    rate_neg = neg[f"alert_{cfg}"].mean()
    thr_gb = np.quantile(neg.gb_full, 1 - rate_neg) if rate_neg > 0 else 1.0       # GB threshold with the same false-alert burden
    for label, alert in [(f"Agent ({cfg}) own alert", pos[f"alert_{cfg}"] == 1), ("GB, full @ same alert burden", pos.gb_full >= thr_gb)]:
        pp = pos.assign(a=alert.values)
        first = pp[pp.a].groupby("icustay_id").hr.min()
        lead = (pp.groupby("icustay_id").onset_hr.first() - first).dropna()
        hit12 = pp[(pp.onset_hr - pp.hr <= HORIZON_H)].groupby("icustay_id").a.any()
        traj_out.append({"method": label, "septic stays": pp.icustay_id.nunique(),
                         "any alert before onset %": round(100 * pp.groupby("icustay_id").a.any().mean(), 1),
                         "alert in final 12 h %": round(100 * hit12.mean(), 1),
                         "median lead time h (IQR)": f"{lead.median():.0f} ({lead.quantile(.25):.0f}-{lead.quantile(.75):.0f})" if len(lead) else "-",
                         "non-septic alerts per patient-day": round((rate_neg if label.startswith("Agent") else (neg.gb_full >= thr_gb).mean()) * ALERTS_PER_DAY, 2)})
if traj_out:
    traj_tab = pd.DataFrame(traj_out).set_index("method")
    print("\nTRAJECTORIES:"); print(traj_tab.to_string()); traj_tab.to_csv(f"{OUT_DIR}/ext_trajectories.csv")


In [ ]:
# =====================================================================================
# PART 15. Generalisability
# Temporal validation across charting systems, leave-one-care-unit-out validation,
# subgroup analysis on the 568 points.
# =====================================================================================
# ---- 15a. temporal validation ----
era_rows = []
for tr_era, te_era in [("CareVue", "MetaVision"), ("MetaVision", "CareVue")]:
    tr = DPF[DPF.era == tr_era]; te = DPF[DPF.era == te_era]
    for g in FEATSETS:
        t0 = time.time()
        pred = fit_calibrated(tr, FEATSETS[g], GB_PARAMS[g])(te)
        internal = te[["icustay_id", "hr", "y"]].merge(ALLP[["icustay_id", "hr", g]], on=["icustay_id", "hr"])  # pooled-CV OOF, same stays
        era_rows.append({"train -> test": f"{tr_era} -> {te_era}", "model": GB_NAMES[g],
                         "test points": len(te), "test positive %": round(100 * te.y.mean(), 2),
                         "AUROC temporal": round(wauc(te.y, pred), 3), "AUROC pooled CV (same stays)": round(wauc(internal.y, internal[g]), 3),
                         "AUPRC temporal": round(wap(te.y, pred), 3),
                         **{f"{k} temporal": v for k, v in calib_stats(te.y, pred).items() if k in ("O/E", "slope", "Brier")}})
        print(f"  {tr_era} -> {te_era} {g} ({(time.time()-t0)/60:.1f} min)", flush=True)
    for s in SCORES:
        era_rows.append({"train -> test": f"(score) {te_era}", "model": SCORE_NAMES[s], "test points": len(te),
                         "test positive %": round(100 * te.y.mean(), 2), "AUROC temporal": round(wauc(te.y, te[s]), 3),
                         "AUPRC temporal": round(wap(te.y, te[s]), 3)})
era_tab = pd.DataFrame(era_rows)
print("Stays recorded in both systems are excluded."); print(era_tab.to_string(index=False))
era_tab.to_csv(f"{OUT_DIR}/ext_temporal_validation.csv", index=False)

# ---- 15a'. internal-external validation: leave one care unit out (gb_full; about 4 model fits per unit) ----
RUN_UNIT_CV = True
if RUN_UNIT_CV:
    unit_rows = []
    for u in sorted(DPF.first_careunit.dropna().unique()):
        tr, te = DPF[DPF.first_careunit != u], DPF[DPF.first_careunit == u]
        if te.y.sum() < 20: continue
        t0 = time.time()
        pred = fit_calibrated(tr, FEATSETS["gb_full"], GB_PARAMS["gb_full"], k=3)(te)
        internal = te[["icustay_id", "hr", "y"]].merge(ALLP[["icustay_id", "hr", "gb_full"]], on=["icustay_id", "hr"])
        unit_rows.append({"held-out unit": u, "test stays": te.icustay_id.nunique(), "test points": len(te),
                          "positive %": round(100 * te.y.mean(), 2),
                          "AUROC held-out": round(wauc(te.y, pred), 3), "AUROC pooled CV (same stays)": round(wauc(internal.y, internal.gb_full), 3),
                          "AUPRC held-out": round(wap(te.y, pred), 3), "SOFA AUROC": round(wauc(te.y, te.sofa), 3), "NEWS2 AUROC": round(wauc(te.y, te.news2), 3),
                          **{f"{k} held-out": v for k, v in calib_stats(te.y, pred).items() if k in ("O/E", "slope")}})
        print(f"  {u} ({(time.time()-t0)/60:.1f} min)", flush=True)
    unit_tab = pd.DataFrame(unit_rows)
    print(unit_tab.to_string(index=False)); unit_tab.to_csv(f"{OUT_DIR}/ext_unit_validation.csv", index=False)

# ---- 15b. subgroups on the 568 points (unweighted AUROC; groups with < 10 positives are not estimated) ----
E["age_group"] = np.where(E.age >= 65, "≥65 y", "<65 y")
E["sex"] = np.where(E.male == 1, "male", "female")
sub_rows = []
for var in ["era", "first_careunit", "sex", "age_group"]:
    for lev, g in E.groupby(var):
        row = {"subgroup": var, "level": lev, "points": len(g), "positives": int(g.y.sum())}
        for k, c in {"Agent": "risk_main", "GB, full": "gb_full", "SOFA": "sofa", "NEWS2": "news2"}.items():
            row[f"AUROC {k}"] = round(wauc(g.y, g[c]), 3) if g.y.sum() >= 10 and (g.y == 0).sum() >= 10 else np.nan
        sub_rows.append(row)
sub_tab = pd.DataFrame(sub_rows)
print(sub_tab.to_string(index=False)); sub_tab.to_csv(f"{OUT_DIR}/ext_subgroups.csv", index=False)


In [ ]:
# =====================================================================================
# PART 16. Collect result tables
# Writes all ext_*.csv tables into one workbook.
# =====================================================================================
# ---- Part 16. Collect the extended results ----
import glob
out_xlsx = f"{OUT_DIR}/extended_results.xlsx"
csvs = sorted(glob.glob(f"{OUT_DIR}/ext_*.csv"))
try:
    with pd.ExcelWriter(out_xlsx) as xw:
        for f in csvs:
            pd.read_csv(f).to_excel(xw, sheet_name=os.path.basename(f)[4:-4][:31], index=False)
    print(f"{len(csvs)} tables -> {out_xlsx}")
except ImportError:
    print("openpyxl not installed (pip install openpyxl); the CSV files are:", *csvs, sep="\n  ")
print("figures:", glob.glob(f"{OUT_DIR}/fig_*.png"))
